In [ ]:
# In[1]:

import pandas as pd
import pytz

# ensure timezone variable (UTC+8) is set as required (not applied to timestamps)
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to build per-component summaries (limit components to top 20 by row count)
def build_component_summary(df, name_col, value_col, top_components=20, top_values=20):
    if df.empty:
        return {
            'global_min_timestamp': None,
            'global_max_timestamp': None,
            'components': []
        }
    global_min = int(df['timestamp'].min())
    global_max = int(df['timestamp'].max())
    counts = df.groupby('cmdb_id').size().rename('row_count')
    unique_vals = df.groupby('cmdb_id')[value_col].unique()
    summary_rows = []
    # order components by row_count desc, then cmdb_id
    ordered = counts.reset_index().sort_values(['row_count','cmdb_id'], ascending=[False, True]).head(top_components)
    for _, row in ordered.iterrows():
        cmdb = row['cmdb_id']
        row_count = int(row['row_count'])
        vals = unique_vals.get(cmdb, [])
        # ensure Python list and limit to top_values
        vals_list = list(vals)[:top_values]
        summary_rows.append({
            'cmdb_id': cmdb,
            'row_count': row_count,
            f'{value_col}_examples': vals_list
        })
    return {
        'global_min_timestamp': global_min,
        'global_max_timestamp': global_max,
        'components': summary_rows
    }

# Metric summary
summary_metric = build_component_summary(df_metric, 'cmdb_id', 'kpi_name', top_components=20, top_values=20)

# Trace summary
summary_trace = build_component_summary(df_trace, 'cmdb_id', 'trace_name', top_components=20, top_values=20)

# Log summary
summary_log = build_component_summary(df_log, 'cmdb_id', 'log_name', top_components=20, top_values=20)

# Error logs summary
if df_error.empty:
    summary_error = {
        'total_row_count': 0,
        'distinct_cmdb_id': [],
        'min_timestamp': None,
        'max_timestamp': None,
        'top_5_rows': []
    }
else:
    total_rows = int(len(df_error))
    distinct_cmdb = sorted(df_error['cmdb_id'].unique())[:20]
    min_ts = int(df_error['timestamp'].min())
    max_ts = int(df_error['timestamp'].max())
    top5_df = df_error[['timestamp','cmdb_id','message']].head(5)
    # Convert top5 to serializable list of dicts with native Python types
    top_5_rows = []
    for _, r in top5_df.iterrows():
        top_5_rows.append({
            'timestamp': int(r['timestamp']),
            'cmdb_id': r['cmdb_id'],
            'message': r['message']
        })
    summary_error = {
        'total_row_count': total_rows,
        'distinct_cmdb_id': distinct_cmdb,
        'min_timestamp': min_ts,
        'max_timestamp': max_ts,
        'top_5_rows': top_5_rows
    }

# Display compact JSON-serializable summaries (variables shown as requested)
summary_metric, summary_trace, summary_log, summary_error

```
Out[1]:
```
summary = (
    "Summary of telemetry files (based on the execution results):\n\n"
    "1) metric.csv\n"
    "- Global min timestamp: 1616481000, max timestamp: 1616482800.\n"
    f"- Components summarized (top {len(summary_metric['components'])} shown): "
    + ", ".join([c['cmdb_id'] for c in summary_metric['components']]) + ".\n"
    "- Top components by row_count (name: rows): "
    + ", ".join([f\"{c['cmdb_id']}: {c['row_count']}\" for c in summary_metric['components'][:5]]) + ".\n"
    "- Example KPI names are included per component (up to 20 each). \n\n"
    "2) trace.csv\n"
    "- Global min timestamp: 1616460, max timestamp: 1616460 (single timestamp in this file).\n"
    f"- Components summarized (top {len(summary_trace['components'])} shown): "
    + ", ".join([c['cmdb_id'] for c in summary_trace['components']]) + ".\n"
    "- Top components by row_count (name: rows): "
    + ", ".join([f\"{c['cmdb_id']}: {c['row_count']}\" for c in summary_trace['components'][:5]]) + ".\n"
    "- Example trace names are included per component (up to 20 each).\n\n"
    "3) log.csv\n"
    "- Global min timestamp: 1616481000, max timestamp: 1616482800.\n"
    f"- Components summarized (top {len(summary_log['components'])} shown): "
    + ", ".join([c['cmdb_id'] for c in summary_log['components']]) + ".\n"
    "- Top components by row_count (name: rows): "
    + ", ".join([f\"{c['cmdb_id']}: {c['row_count']}\" for c in summary_log['components'][:5]]) + ".\n"
    "- Example log names are included per component (up to 20 each).\n\n"
    "4) error_logs.csv\n"
    "- Total row count: 0 (no error log rows found).\n"
    "- Distinct cmdb_id: none.\n"
    "- No timestamps or top rows to show.\n\n"
    "Notes: All timestamps are kept as raw integer seconds (UTC+8 context). The per-component example lists contain up to 20 distinct KPI/trace/log names as requested."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

({'global_min_timestamp': 1616481000, 'global_max_timestamp': 1616482800, 'components': [{'cmdb_id': 'Mysql01', 'row_count': 3746, 'kpi_name_examples': ['OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdc_DSKTps', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'Mysql-MySQL_3306_Aborted Clients', 'Mysql-MySQL_3306_Aborted Connects', 'Mysql-MySQL_3306_Binlog Cache Disk Use', 'Mysql-MySQL_3306_Binlog Cache Use', 'Mysql-MySQL_3306_Binlog stmt cache disk use', 'Mysql-MySQL_3306_Binlog stmt cache use', 'Mysql-MySQL_3306_Bytes Received', 'Mysql-MySQL_3306_Bytes Sent', 'Mysql-MySQL_3306_Com Delete', 'Mysql-MySQL_3306_Com Delete Multi', 'Mysql-MySQL_3306_Com Insert', 'Mysql-MySQL_3306_Com Insert Select', 'Mysql-MySQL_3306_Com Load']}, {'cmdb_id': 'Mysql02', 'row_count': 3715, 'kpi_name_examples': ['OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'Mysql-MySQL_3306_Aborted Clients', 'Mysql-MySQL_3306_Aborted Connects', 'Mysql-MySQL_3306_Binlog Cache Disk Use', 'Mysql-MySQL_3306_Binlog Cache Use', 'Mysql-MySQL_3306_Binlog stmt cache disk use', 'Mysql-MySQL_3306_Binlog stmt cache use', 'Mysql-MySQL_3306_Bytes Received', 'Mysql-MySQL_3306_Bytes Sent', 'Mysql-MySQL_3306_Com Delete', 'Mysql-MySQL_3306_Com Delete Multi', 'Mysql-MySQL_3306_Com Insert', 'Mysql-MySQL_3306_Com Insert Select', 'Mysql-MySQL_3306_Com Load', 'Mysql-MySQL_3306_Com Replace']}, {'cmdb_id': 'Tomcat03', 'row_count': 2061, 'kpi_name_examples': ['OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----tmp_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----tmp_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----tomcat_FSCapacity']}, {'cmdb_id': 'Tomcat02', 'row_count': 2034, 'kpi_name_examples': ['OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----tmp_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----tmp_FSInodeUsedPercent']}, {'cmdb_id': 'Tomcat04', 'row_count': 2016, 'kpi_name_examples': ['OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----tmp_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----tmp_FSInodeUsedPercent']}, {'cmdb_id': 'Tomcat01', 'row_count': 1953, 'kpi_name_examples': ['OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----cmbc_admin_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----home_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----home_FSInodeUsedPercent']}, {'cmdb_id': 'MG02', 'row_count': 1618, 'kpi_name_examples': ['JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'JVM-Operating System_7779_JVM_JVM_CPULoad', 'JVM-Runtime_7779_JVM_JVM_Uptime', 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_FILESYSTEM_----_FSCapacity', 'OSLinux-OSLinux_FILESYSTEM_----_FSInodeUsedPercent', 'OSLinux-OSLinux_FILESYSTEM_----boot_FSCapacity']}, {'cmdb_id': 'MG01', 'row_count': 1609, 'kpi_name_examples': ['JVM-Memory_7779_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7779_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed', 'JVM-Operating System_7779_JVM_JVM_CPULoad', 'JVM-Runtime_7779_JVM_JVM_Uptime', 'JVM-Threads_7779_JVM_ThreadCount_Threads', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-0_SingleCpuidle', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuidle', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuidle', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuidle', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio']}, {'cmdb_id': 'Redis02', 'row_count': 1151, 'kpi_name_examples': ['OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-NTP_197.30.1.67_NtpServerTimeOffset', 'OSLinux-NTP_197.30.1.68_NtpServerTimeOffset', 'OSLinux-OSLinux_FILE_-home-zabbix_DirSizeMB', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKAvgServ', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKAvgServ', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_UserMem', 'OSLinux-OSLinux_NETWORK_NETWORK_TotalTcpConnNum', 'OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU_CPULoad', 'OSLinux-CPU_CPU_CPUSysTime', 'OSLinux-CPU_CPU_CPUUserTime', 'OSLinux-CPU_CPU_CPUWio', 'OSLinux-CPU_CPU_CPUidleutil', 'OSLinux-OSLinux_FILE_-tmp-zabbix_agentd.log_FileSizeMB', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPercentBusy']}, {'cmdb_id': 'IG01', 'row_count': 333, 'kpi_name_examples': ['JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Operating System_7778_JVM_JVM_CPULoad', 'JVM-Runtime_7778_JVM_JVM_Uptime', 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite']}, {'cmdb_id': 'IG02', 'row_count': 325, 'kpi_name_examples': ['JVM-Memory_7778_JVM_Memory_HeapMemoryMax', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsed', 'JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed', 'JVM-Operating System_7778_JVM_JVM_CPULoad', 'JVM-Runtime_7778_JVM_JVM_Uptime', 'JVM-Threads_7778_JVM_ThreadCount_Threads', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'JVM-Memory_7778_JVM_Memory_HeapMemoryUsage', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil']}, {'cmdb_id': 'apache01', 'row_count': 158, 'kpi_name_examples': ['OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps']}, {'cmdb_id': 'Redis01', 'row_count': 132, 'kpi_name_examples': ['OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc', 'OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc', 'OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps']}, {'cmdb_id': 'apache02', 'row_count': 125, 'kpi_name_examples': ['OSLinux-CPU_CPU_CPUCpuUtil', 'OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc', 'OSLinux-CPU_CPU-0_SingleCpuUtil', 'OSLinux-CPU_CPU-1_SingleCpuUtil', 'OSLinux-CPU_CPU-2_SingleCpuUtil', 'OSLinux-CPU_CPU-3_SingleCpuUtil', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKReadWrite', 'OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps', 'OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc']}, {'cmdb_id': 'ServiceTest1', 'row_count': 124, 'kpi_name_examples': ['app.cnt', 'app.mrt', 'app.rr', 'app.sr']}, {'cmdb_id': 'ServiceTest10', 'row_count': 124, 'kpi_name_examples': ['app.cnt', 'app.mrt', 'app.rr', 'app.sr']}, {'cmdb_id': 'ServiceTest11', 'row_count': 124, 'kpi_name_examples': ['app.cnt', 'app.mrt', 'app.rr', 'app.sr']}, {'cmdb_id': 'ServiceTest2', 'row_count': 124, 'kpi_name_examples': ['app.cnt', 'app.mrt', 'app.rr', 'app.sr']}, {'cmdb_id': 'ServiceTest3', 'row_count': 124, 'kpi_name_examples': ['app.cnt', 'app.mrt', 'app.rr', 'app.sr']}, {'cmdb_id': 'ServiceTest4', 'row_count': 124, 'kpi_name_examples': ['app.cnt', 'app.mrt', 'app.rr', 'app.sr']}]}, {'global_min_timestamp': 1616460, 'global_max_timestamp': 1616460, 'components': [{'cmdb_id': 'MG01', 'row_count': 42, 'trace_name_examples': ['trace.from_Tomcat01.duration_mean', 'trace.from_Tomcat01.duration_p95', 'trace.from_Tomcat01.row_count', 'trace.from_Tomcat02.duration_mean', 'trace.from_Tomcat02.duration_p95', 'trace.from_Tomcat02.row_count', 'trace.from_Tomcat03.duration_mean', 'trace.from_Tomcat03.duration_p95', 'trace.from_Tomcat03.row_count', 'trace.from_Tomcat04.duration_mean', 'trace.from_Tomcat04.duration_p95', 'trace.from_Tomcat04.row_count', 'trace.from_dockerA1.duration_mean', 'trace.from_dockerA1.duration_p95', 'trace.from_dockerA1.row_count', 'trace.from_dockerA2.duration_mean', 'trace.from_dockerA2.duration_p95', 'trace.from_dockerA2.row_count', 'trace.from_dockerB1.duration_mean', 'trace.from_dockerB1.duration_p95']}, {'cmdb_id': 'MG02', 'row_count': 42, 'trace_name_examples': ['trace.from_Tomcat01.duration_mean', 'trace.from_Tomcat01.duration_p95', 'trace.from_Tomcat01.row_count', 'trace.from_Tomcat02.duration_mean', 'trace.from_Tomcat02.duration_p95', 'trace.from_Tomcat02.row_count', 'trace.from_Tomcat03.duration_mean', 'trace.from_Tomcat03.duration_p95', 'trace.from_Tomcat03.row_count', 'trace.from_Tomcat04.duration_mean', 'trace.from_Tomcat04.duration_p95', 'trace.from_Tomcat04.row_count', 'trace.from_dockerA1.duration_mean', 'trace.from_dockerA1.duration_p95', 'trace.from_dockerA1.row_count', 'trace.from_dockerA2.duration_mean', 'trace.from_dockerA2.duration_p95', 'trace.from_dockerA2.row_count', 'trace.from_dockerB1.duration_mean', 'trace.from_dockerB1.duration_p95']}, {'cmdb_id': 'IG01', 'row_count': 18, 'trace_name_examples': ['trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_Tomcat01.duration_mean', 'trace.to_Tomcat01.duration_p95', 'trace.to_Tomcat01.row_count', 'trace.to_Tomcat02.duration_mean', 'trace.to_Tomcat02.duration_p95', 'trace.to_Tomcat02.row_count', 'trace.to_Tomcat03.duration_mean', 'trace.to_Tomcat03.duration_p95', 'trace.to_Tomcat03.row_count', 'trace.to_Tomcat04.duration_mean', 'trace.to_Tomcat04.duration_p95', 'trace.to_Tomcat04.row_count']}, {'cmdb_id': 'IG02', 'row_count': 18, 'trace_name_examples': ['trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_Tomcat01.duration_mean', 'trace.to_Tomcat01.duration_p95', 'trace.to_Tomcat01.row_count', 'trace.to_Tomcat02.duration_mean', 'trace.to_Tomcat02.duration_p95', 'trace.to_Tomcat02.row_count', 'trace.to_Tomcat03.duration_mean', 'trace.to_Tomcat03.duration_p95', 'trace.to_Tomcat03.row_count', 'trace.to_Tomcat04.duration_mean', 'trace.to_Tomcat04.duration_p95', 'trace.to_Tomcat04.row_count']}, {'cmdb_id': 'Tomcat01', 'row_count': 18, 'trace_name_examples': ['trace.from_IG01.duration_mean', 'trace.from_IG01.duration_p95', 'trace.from_IG01.row_count', 'trace.from_IG02.duration_mean', 'trace.from_IG02.duration_p95', 'trace.from_IG02.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count']}, {'cmdb_id': 'Tomcat02', 'row_count': 18, 'trace_name_examples': ['trace.from_IG01.duration_mean', 'trace.from_IG01.duration_p95', 'trace.from_IG01.row_count', 'trace.from_IG02.duration_mean', 'trace.from_IG02.duration_p95', 'trace.from_IG02.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count']}, {'cmdb_id': 'Tomcat03', 'row_count': 18, 'trace_name_examples': ['trace.from_IG01.duration_mean', 'trace.from_IG01.duration_p95', 'trace.from_IG01.row_count', 'trace.from_IG02.duration_mean', 'trace.from_IG02.duration_p95', 'trace.from_IG02.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count']}, {'cmdb_id': 'Tomcat04', 'row_count': 18, 'trace_name_examples': ['trace.from_IG01.duration_mean', 'trace.from_IG01.duration_p95', 'trace.from_IG01.row_count', 'trace.from_IG02.duration_mean', 'trace.from_IG02.duration_p95', 'trace.from_IG02.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count']}, {'cmdb_id': 'dockerA1', 'row_count': 18, 'trace_name_examples': ['trace.from_MG01.duration_mean', 'trace.from_MG01.duration_p95', 'trace.from_MG01.row_count', 'trace.from_MG02.duration_mean', 'trace.from_MG02.duration_p95', 'trace.from_MG02.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count']}, {'cmdb_id': 'dockerA2', 'row_count': 18, 'trace_name_examples': ['trace.from_MG01.duration_mean', 'trace.from_MG01.duration_p95', 'trace.from_MG01.row_count', 'trace.from_MG02.duration_mean', 'trace.from_MG02.duration_p95', 'trace.from_MG02.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count']}, {'cmdb_id': 'dockerB1', 'row_count': 18, 'trace_name_examples': ['trace.from_MG01.duration_mean', 'trace.from_MG01.duration_p95', 'trace.from_MG01.row_count', 'trace.from_MG02.duration_mean', 'trace.from_MG02.duration_p95', 'trace.from_MG02.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count']}, {'cmdb_id': 'dockerB2', 'row_count': 18, 'trace_name_examples': ['trace.from_MG01.duration_mean', 'trace.from_MG01.duration_p95', 'trace.from_MG01.row_count', 'trace.from_MG02.duration_mean', 'trace.from_MG02.duration_p95', 'trace.from_MG02.row_count', 'trace.from_root.duration_mean', 'trace.from_root.duration_p95', 'trace.from_root.row_count', 'trace.self.duration_mean', 'trace.self.duration_p95', 'trace.self.row_count', 'trace.to_MG01.duration_mean', 'trace.to_MG01.duration_p95', 'trace.to_MG01.row_count', 'trace.to_MG02.duration_mean', 'trace.to_MG02.duration_p95', 'trace.to_MG02.row_count']}]}, {'global_min_timestamp': 1616481000, 'global_max_timestamp': 1616482800, 'components': [{'cmdb_id': 'Tomcat04', 'row_count': 87, 'log_name_examples': ['log.error_count', 'log.gc_pause_max', 'log.row_count']}, {'cmdb_id': 'Tomcat01', 'row_count': 86, 'log_name_examples': ['log.error_count', 'log.gc_pause_max', 'log.row_count']}, {'cmdb_id': 'Tomcat02', 'row_count': 86, 'log_name_examples': ['log.error_count', 'log.row_count', 'log.gc_pause_max']}, {'cmdb_id': 'Tomcat03', 'row_count': 86, 'log_name_examples': ['log.error_count', 'log.gc_pause_max', 'log.row_count']}, {'cmdb_id': 'apache01', 'row_count': 62, 'log_name_examples': ['log.error_count', 'log.row_count']}, {'cmdb_id': 'apache02', 'row_count': 62, 'log_name_examples': ['log.error_count', 'log.row_count']}, {'cmdb_id': 'Mysql02', 'row_count': 2, 'log_name_examples': ['log.error_count', 'log.row_count']}]}, {'total_row_count': 0, 'distinct_cmdb_id': [], 'min_timestamp': None, 'max_timestamp': None, 'top_5_rows': []})```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
from datetime import datetime

# reuse existing df_metric loaded earlier
# If df_metric not present in this kernel state, load it (safe fallback)
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# 1) Allowed root-cause components
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

df_allowed = df_metric[df_metric['cmdb_id'].isin(allowed_components)].copy()

# 2) Compute global thresholds (P95, P5) and total point count per (cmdb_id, kpi_name)
grouped = df_allowed.groupby(['cmdb_id', 'kpi_name'])['value']
thresholds = grouped.agg(
    global_count='size',
    global_P95=lambda x: float(x.quantile(0.95)),
    global_P5=lambda x: float(x.quantile(0.05))
).reset_index()

total_series_analyzed = thresholds.shape[0]  # number of component-kpi series analyzed

# 3) Define incident window in UTC+8 and convert to epoch seconds
tz = pytz.timezone('Asia/Shanghai')
start_dt = tz.localize(datetime(2021, 3, 23, 14, 30, 0))
end_dt = tz.localize(datetime(2021, 3, 23, 15, 0, 0))
start_ts = int(start_dt.timestamp())
end_ts = int(end_dt.timestamp())

# Filter df_allowed to the incident window
df_window = df_allowed[(df_allowed['timestamp'] >= start_ts) & (df_allowed['timestamp'] <= end_ts)].copy()

# If no rows in window, prepare empty outputs
if df_window.empty:
    result_df_top = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count_in_window','earliest_timestamp_in_window',
        'max_value_in_window','min_value_in_window','mean_value_in_window',
        'global_P95','global_P5','count_above_P95_in_window','count_below_P5_in_window'
    ])
    meta = {
        'total_component_kpi_series_analyzed': int(total_series_analyzed),
        'incident_window_start_ts': start_ts,
        'incident_window_end_ts': end_ts
    }
    result_df_top, meta
else:
    # 3 (continued): attach global thresholds to window rows for flagging
    df_win_with_thresh = df_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # Flags for breaches relative to global thresholds
    df_win_with_thresh['above_P95'] = (df_win_with_thresh['value'] > df_win_with_thresh['global_P95']).astype(int)
    df_win_with_thresh['below_P5'] = (df_win_with_thresh['value'] < df_win_with_thresh['global_P5']).astype(int)

    # Aggregate per component-kpi within the window
    agg = df_win_with_thresh.groupby(['cmdb_id','kpi_name']).agg(
        count_in_window=('value','size'),
        earliest_timestamp_in_window=('timestamp','min'),
        max_value_in_window=('value','max'),
        min_value_in_window=('value','min'),
        mean_value_in_window=('value','mean'),
        count_above_P95_in_window=('above_P95','sum'),
        count_below_P5_in_window=('below_P5','sum'),
    ).reset_index()

    # Merge global thresholds and global_count back in
    agg = agg.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # 4) Filter only rows with breaches (above P95 or below P5)
    breached = agg[(agg['count_above_P95_in_window'] > 0) | (agg['count_below_P5_in_window'] > 0)].copy()

    if breached.empty:
        result_df_top = pd.DataFrame(columns=[
            'cmdb_id','kpi_name','count_in_window','earliest_timestamp_in_window',
            'max_value_in_window','min_value_in_window','mean_value_in_window',
            'global_P95','global_P5','count_above_P95_in_window','count_below_P5_in_window'
        ])
    else:
        # severity metric: primary sort - count_above desc, then count_below desc, then abs(max_value - global_P95) desc
        breached['severity_gap'] = (breached['max_value_in_window'] - breached['global_P95']).abs()
        breached_sorted = breached.sort_values(
            by=['count_above_P95_in_window','count_below_P5_in_window','severity_gap'],
            ascending=[False, False, False]
        )

        # Select and rename columns to match requested names and keep them compact
        select_cols = [
            'cmdb_id','kpi_name','count_in_window','earliest_timestamp_in_window',
            'max_value_in_window','min_value_in_window','mean_value_in_window',
            'global_P95','global_P5','count_above_P95_in_window','count_below_P5_in_window'
        ]
        result_df_top = breached_sorted[select_cols].head(20).copy()

        # Ensure numeric fields are native Python types where possible for JSON-serializability
        # (but we'll keep as DataFrame for compact display per instructions)

    meta = {
        'total_component_kpi_series_analyzed': int(total_series_analyzed),
        'incident_window_start_ts': int(start_ts),
        'incident_window_end_ts': int(end_ts)
    }

    # Final displayed outputs: top breach candidates (DataFrame) and meta summary (dict)
    result_df_top, meta

```
Out[2]:
```
Summary of the metric analysis (plain English):

- Incident window used (UTC+8): March 23, 2021 14:30 to 15:00
  - start_ts = 1616484600
  - end_ts   = 1616486400

- Components considered (restricted set): apache01, apache02, Tomcat01, Tomcat02, Tomcat03, Tomcat04, MG01, MG02, IG01, IG02, Mysql01, Mysql02, Redis01, Redis02.

- Result: There were zero metric data points from the allowed components inside the incident window (no rows matched timestamp between 1616484600 and 1616486400). 
  -> Therefore no component–KPI series produced any P95/P5 breaches in that window (no count_above_P95_in_window > 0 and no count_below_P5_in_window > 0).

- Conclusion: With the available metric data, no candidate breaches were detected for the given incident window; you may need to (a) verify the intended incident time window, or (b) check whether metric collection covers that time period.

The original code execution output of IPython Kernel is also provided below for reference:

None```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
from datetime import datetime

# reuse df_trace if present, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')

# ensure types
df_trace['timestamp'] = df_trace['timestamp'].astype(int)

# timezone (UTC+8) as required (for internal use/reference)
tz = pytz.timezone('Asia/Shanghai')

# 1) Global min/max and total row count
global_min_ts = int(df_trace['timestamp'].min()) if not df_trace.empty else None
global_max_ts = int(df_trace['timestamp'].max()) if not df_trace.empty else None
total_row_count = int(df_trace.shape[0])

trace_meta = {
    'global_min_timestamp': global_min_ts,
    'global_max_timestamp': global_max_ts,
    'total_row_count': total_row_count
}

# 2) Count rows for allowed components
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

df_allowed_trace = df_trace[df_trace['cmdb_id'].isin(allowed_components)].copy()
allowed_rows_count = int(df_allowed_trace.shape[0])

# 3) Incident window filtering (use provided epoch seconds)
start_ts = 1616484600
end_ts = 1616486400

df_window = df_allowed_trace[(df_allowed_trace['timestamp'] >= start_ts) & (df_allowed_trace['timestamp'] <= end_ts)].copy()

count_in_window = int(df_window.shape[0])

# distinct timestamps and cmdb_ids in window (limit lists)
distinct_timestamps = sorted(df_window['timestamp'].unique())[:20]
distinct_timestamps = [int(x) for x in distinct_timestamps]

distinct_cmdbs = sorted(df_window['cmdb_id'].unique())[:20]

# For each cmdb_id present in the window: up to 10 distinct trace_name examples and earliest timestamp_in_window
per_component_rows = []
if not df_window.empty:
    grouped = df_window.groupby('cmdb_id')
    for cmdb, g in grouped:
        trace_names = list(pd.Series(g['trace_name'].unique()).sort_values()[:10])
        earliest_ts = int(g['timestamp'].min())
        per_component_rows.append({
            'cmdb_id': cmdb,
            'count_in_window': int(g.shape[0]),
            'earliest_timestamp_in_window': earliest_ts,
            'trace_name_examples': trace_names
        })
    # sort per_component_rows by count_in_window desc, then cmdb_id and limit to 20
    per_component_df = pd.DataFrame(per_component_rows)
    if not per_component_df.empty:
        per_component_df = per_component_df.sort_values(by=['count_in_window','cmdb_id'], ascending=[False, True]).head(20)
        # convert back to list of dicts with native types
        per_component_rows = []
        for _, r in per_component_df.iterrows():
            per_component_rows.append({
                'cmdb_id': r['cmdb_id'],
                'count_in_window': int(r['count_in_window']),
                'earliest_timestamp_in_window': int(r['earliest_timestamp_in_window']),
                'trace_name_examples': r['trace_name_examples']
            })

# Final compact JSON-serializable summary
result_summary = {
    'trace_meta': trace_meta,
    'allowed_components_rows_count': allowed_rows_count,
    'incident_window': {
        'start_ts': int(start_ts),
        'end_ts': int(end_ts),
        'count_in_window': count_in_window,
        'distinct_timestamps_in_window_limited': distinct_timestamps,
        'distinct_cmdb_ids_in_window_limited': distinct_cmdbs
    },
    'per_component_summary_limited': per_component_rows  # up to 20 components, each with up to 10 trace_name examples
}

result_summary

```
Out[3]:
```
Summary of trace.csv check (plain English):

- trace.csv overall:
  - global_min_timestamp = 1616460
  - global_max_timestamp = 1616460
  - total_row_count = 264

- Rows for the allowed components (apache01, apache02, Tomcat01..Tomcat04, MG01, MG02, IG01, IG02, Mysql01, Mysql02, Redis01, Redis02):
  - count = 192 rows

- Incident window (UTC+8): start_ts = 1616484600, end_ts = 1616486400
  - rows in window (for allowed components) = 0
  - no distinct timestamps in window
  - no distinct cmdb_id in window
  - per-component summaries in window = none

Conclusion: trace.csv contains data (264 rows, 192 for allowed components) but all rows have timestamp 1616460 (a single timestamp) which is outside the requested incident window (1616484600–1616486400). Therefore there is no trace data for the allowed components in the incident window.

The original code execution output of IPython Kernel is also provided below for reference:

{'trace_meta': {'global_min_timestamp': 1616460, 'global_max_timestamp': 1616460, 'total_row_count': 264}, 'allowed_components_rows_count': 192, 'incident_window': {'start_ts': 1616484600, 'end_ts': 1616486400, 'count_in_window': 0, 'distinct_timestamps_in_window_limited': [], 'distinct_cmdb_ids_in_window_limited': []}, 'per_component_summary_limited': []}```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz

# Reuse existing DataFrames if present, otherwise load from CSVs
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_trace
except NameError:
    df_trace = pd.read_csv('trace.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')
try:
    df_error
except NameError:
    # file is named error_logs.csv
    df_error = pd.read_csv('error_logs.csv')

# Ensure timestamp column is integer
for df in (df_metric, df_trace, df_log, df_error):
    if 'timestamp' in df.columns:
        df['timestamp'] = df['timestamp'].astype(int)

# Timezone object per rule (UTC+8). We won't convert timestamps but we provide tz variable as required.
tz = pytz.timezone('Asia/Shanghai')

# Allowed components for metric/trace/log; for error_logs we use all rows
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Incident window (epoch seconds) provided
start_ts = 1616484600
end_ts = 1616486400

def summarize_file(df, restrict_to_allowed=True):
    # df: DataFrame with 'timestamp' and 'cmdb_id'
    # restrict_to_allowed: if True, counts in window and before/after row counts are restricted to allowed_components
    # For error_logs set restrict_to_allowed=False
    summary = {}
    if df.empty:
        summary['global_min_timestamp'] = None
        summary['global_max_timestamp'] = None
        summary['total_row_count'] = 0
        summary['count_in_window'] = 0
        summary['timestamps_before'] = []
        summary['timestamps_after'] = []
        summary['distinct_cmdb_ids_in_window_limited'] = []
        summary['per_cmdb_in_window_limited'] = []
        return summary

    global_min = int(df['timestamp'].min())
    global_max = int(df['timestamp'].max())
    total_rows = int(df.shape[0])

    # apply allowed component restriction for window/counts and before/after counts if needed
    if restrict_to_allowed:
        df_allowed = df[df['cmdb_id'].isin(allowed_components)].copy()
    else:
        df_allowed = df  # use all rows

    # count in window (inclusive)
    df_window = df_allowed[(df_allowed['timestamp'] >= start_ts) & (df_allowed['timestamp'] <= end_ts)].copy()
    count_in_window = int(df_window.shape[0])

    # helper to get up to n nearest timestamps before start_ts
    def nearest_before(df_src, n=3):
        ts_unique = sorted(df_src.loc[df_src['timestamp'] < start_ts, 'timestamp'].unique(), reverse=True)
        ts_unique = ts_unique[:n]
        rows = []
        for t in ts_unique:
            cnt = int(df_src.loc[df_src['timestamp'] == t].shape[0])
            rows.append({'timestamp': int(t), 'row_count': cnt})
        return rows

    # helper to get up to n nearest timestamps after end_ts
    def nearest_after(df_src, n=3):
        ts_unique = sorted(df_src.loc[df_src['timestamp'] > end_ts, 'timestamp'].unique())
        ts_unique = ts_unique[:n]
        rows = []
        for t in ts_unique:
            cnt = int(df_src.loc[df_src['timestamp'] == t].shape[0])
            rows.append({'timestamp': int(t), 'row_count': cnt})
        return rows

    timestamps_before = nearest_before(df_allowed, n=3)
    timestamps_after = nearest_after(df_allowed, n=3)

    # If there are rows in window, collect distinct cmdb_ids (limit 20) and earliest timestamp per cmdb_id
    distinct_cmdbs = []
    per_cmdb = []
    if count_in_window > 0:
        distinct_cmdbs = sorted(df_window['cmdb_id'].unique())[:20]
        # earliest timestamp per cmdb_id
        grouped = df_window.groupby('cmdb_id')
        for cmdb in distinct_cmdbs:
            g = grouped.get_group(cmdb)
            per_cmdb.append({
                'cmdb_id': cmdb,
                'earliest_timestamp_in_window': int(g['timestamp'].min()),
                'count_in_window': int(g.shape[0])
            })

    summary.update({
        'global_min_timestamp': int(global_min),
        'global_max_timestamp': int(global_max),
        'total_row_count': int(total_rows),
        'count_in_window': int(count_in_window),
        'timestamps_before': timestamps_before,   # up to 3 nearest before start_ts
        'timestamps_after': timestamps_after,     # up to 3 nearest after end_ts
        'distinct_cmdb_ids_in_window_limited': distinct_cmdbs,
        'per_cmdb_in_window_limited': per_cmdb
    })
    return summary

# Summarize each file per rules (metric/trace/log restricted to allowed components; error_logs not restricted)
summary_metric = summarize_file(df_metric, restrict_to_allowed=True)
summary_trace = summarize_file(df_trace, restrict_to_allowed=True)
summary_log = summarize_file(df_log, restrict_to_allowed=True)
summary_error_logs = summarize_file(df_error, restrict_to_allowed=False)

# Package compact JSON-serializable result
result_summary = {
    'incident_window': {'start_ts': int(start_ts), 'end_ts': int(end_ts)},
    'metric.csv': summary_metric,
    'trace.csv': summary_trace,
    'log.csv': summary_log,
    'error_logs.csv': summary_error_logs
}

# Display the compact summary object (variable output)
result_summary

```
Out[4]:
```
Incident window (UTC+8): start_ts = 1616484600, end_ts = 1616486400.

Summary (per file):

- metric.csv
  - global_min_timestamp: 1616481000
  - global_max_timestamp: 1616482800
  - total_row_count: 22,340
  - count_in_window: 0 (no metric rows inside the incident window)
  - up to 3 nearest timestamps before window (timestamp : row_count):
    - 1616482800 : 428
    - 1616482740 : 728
    - 1616482680 : 703
  - nearest timestamps after window: none

- trace.csv
  - global_min_timestamp: 1616460
  - global_max_timestamp: 1616460
  - total_row_count: 264
  - count_in_window: 0 (no trace rows inside the incident window)
  - nearest timestamps before window:
    - 1616460 : 192
  - nearest timestamps after window: none

- log.csv
  - global_min_timestamp: 1616481000
  - global_max_timestamp: 1616482800
  - total_row_count: 471
  - count_in_window: 0 (no log rows inside the incident window)
  - nearest timestamps before window:
    - 1616482800 : 12
    - 1616482740 : 15
    - 1616482680 : 16
  - nearest timestamps after window: none

- error_logs.csv
  - global_min_timestamp: None
  - global_max_timestamp: None
  - total_row_count: 0
  - count_in_window: 0
  - no timestamps before/after (file empty)

Overall conclusion:
- None of the four telemetry files contain rows for the allowed components within the specified incident window (1616484600–1616486400). There are recent data points before the window in metric and log (and a single earlier timestamp in trace), but no data after the window in these files. Error logs are empty.

The original code execution output of IPython Kernel is also provided below for reference:

{'incident_window': {'start_ts': 1616484600, 'end_ts': 1616486400}, 'metric.csv': {'global_min_timestamp': 1616481000, 'global_max_timestamp': 1616482800, 'total_row_count': 22340, 'count_in_window': 0, 'timestamps_before': [{'timestamp': 1616482800, 'row_count': 428}, {'timestamp': 1616482740, 'row_count': 728}, {'timestamp': 1616482680, 'row_count': 703}], 'timestamps_after': [], 'distinct_cmdb_ids_in_window_limited': [], 'per_cmdb_in_window_limited': []}, 'trace.csv': {'global_min_timestamp': 1616460, 'global_max_timestamp': 1616460, 'total_row_count': 264, 'count_in_window': 0, 'timestamps_before': [{'timestamp': 1616460, 'row_count': 192}], 'timestamps_after': [], 'distinct_cmdb_ids_in_window_limited': [], 'per_cmdb_in_window_limited': []}, 'log.csv': {'global_min_timestamp': 1616481000, 'global_max_timestamp': 1616482800, 'total_row_count': 471, 'count_in_window': 0, 'timestamps_before': [{'timestamp': 1616482800, 'row_count': 12}, {'timestamp': 1616482740, 'row_count': 15}, {'timestamp': 1616482680, 'row_count': 16}], 'timestamps_after': [], 'distinct_cmdb_ids_in_window_limited': [], 'per_cmdb_in_window_limited': []}, 'error_logs.csv': {'global_min_timestamp': None, 'global_max_timestamp': None, 'total_row_count': 0, 'count_in_window': 0, 'timestamps_before': [], 'timestamps_after': [], 'distinct_cmdb_ids_in_window_limited': [], 'per_cmdb_in_window_limited': []}}```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# reuse df_metric if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types and timezone object as required
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
tz = pytz.timezone('Asia/Shanghai')

# Allowed components
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# 1) Compute global thresholds (P95, P5) and global_count per (cmdb_id, kpi_name) using all rows (but restricted to allowed components)
df_allowed = df_metric[df_metric['cmdb_id'].isin(allowed_components)].copy()

thresholds = df_allowed.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_count='size',
    global_P95=lambda s: float(s.quantile(0.95)),
    global_P5 = lambda s: float(s.quantile(0.05))
).reset_index()

number_of_component_kpi_series_examined = int(thresholds.shape[0])

# 2) Restrict rows to the three nearest-before-window timestamps
target_timestamps = [1616482680, 1616482740, 1616482800]
df_targets = df_allowed[df_allowed['timestamp'].isin(target_timestamps)].copy()

# 3) Attach global thresholds to each target row
df_targets = df_targets.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

# 4) Identify breaches where value > global_P95 or value < global_P5
if df_targets.empty:
    # prepare empty results
    result_df_breaches = pd.DataFrame(columns=[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','global_count','severity'
    ])
    number_of_breach_rows_found = 0
else:
    df_targets['above_P95'] = df_targets['value'] > df_targets['global_P95']
    df_targets['below_P5'] = df_targets['value'] < df_targets['global_P5']
    # severity: for above use value - P95, for below use P5 - value (both positive); take whichever applies
    df_targets['severity'] = np.where(
        df_targets['above_P95'],
        (df_targets['value'] - df_targets['global_P95']).abs(),
        np.where(df_targets['below_P5'],
                 (df_targets['global_P5'] - df_targets['value']).abs(),
                 0.0)
    )
    df_breaches = df_targets[(df_targets['above_P95']) | (df_targets['below_P5'])].copy()
    number_of_breach_rows_found = int(df_breaches.shape[0])
    if df_breaches.empty:
        result_df_breaches = pd.DataFrame(columns=[
            'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','global_count','severity'
        ])
    else:
        # sort by severity desc and limit top 20
        df_breaches_sorted = df_breaches.sort_values(by='severity', ascending=False).head(20)
        # select required columns and ensure compact ordering
        result_df_breaches = df_breaches_sorted[[
            'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','global_count','severity'
        ]].reset_index(drop=True)

# Summary small dict
summary = {
    'number_of_component_kpi_series_examined': number_of_component_kpi_series_examined,
    'number_of_breach_rows_found': number_of_breach_rows_found,
    'examined_timestamps_limited': target_timestamps
}

# Return compact DataFrame of breaches (top 20) and summary
result_df_breaches, summary

```
Out[5]:
```
Summary of the focused metric check (plain English):

- Scope: metric.csv restricted to allowed components (apache01, apache02, Tomcat01–Tomcat04, MG01, MG02, IG01, IG02, Mysql01, Mysql02, Redis01, Redis02). Global P95/P5 were computed per (cmdb_id, kpi_name) across all available rows.

- Examined timestamps: only the three nearest-before-window timestamps [1616482680, 1616482740, 1616482800].

- Counts:
  - Number of distinct component–KPI series examined: 1,110
  - Number of breach rows found at those three timestamps (value > global_P95 OR value < global_P5): 180

- Breach types and top examples (top 20 by severity were returned):
  - Memory low-value breach: IG01 – JVM-Memory_HeapMemoryUsed at 1616482680 had value ~8.80e7 which is below its global P5 (~1.07e8).
  - Tomcat JVM/free-memory: Tomcat02 – JVMFreeMemory at 1616482680 exceeded its global P95.
  - JVM heap and network packet metrics on MG01/MG02 showed breaches (e.g., JVM heap, NETPacketsIn/Out).
  - Several Tomcat instances (Tomcat01–Tomcat04) had Request/Processing metrics at these timestamps exceeding their global P95.
  - Many network packet counters (OSLinux_network NETPacketsIn/NETPacketsOut) for MG01/MG02/Tomcat* are among the breaches.

- Output content (what was returned):
  - For each breach row returned: timestamp, cmdb_id, kpi_name, value, global_P95, global_P5, global_count (number of historical points used to compute thresholds), and a severity score (absolute deviation).
  - Results were limited to the top 20 breach rows sorted by severity (largest deviation first).

- Conclusion / next steps:
  - There are non-trivial breaches at the three pre-window timestamps across memory, request-processing, and network KPIs for several allowed components (notably IG01, MG01/MG02, and Tomcat instances).
  - If you want the full list of all 180 breach rows or further drill-down by component/KPI/time, I can provide it.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp   cmdb_id                                           kpi_name         value    global_P95     global_P5  global_count     severity
0   1616482680      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.799224e+07  8.498951e+08  1.066455e+08            31  18653308.00
1   1616482680  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory  2.699068e+09  2.686177e+09  2.466088e+09             4  12891583.20
2   1616482740      MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed  1.011772e+09  1.009610e+09  2.690440e+08            31   2162248.00
3   1616482800  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...  9.369277e+08  9.364945e+08  9.295056e+08            31    433188.00
4   1616482800  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.668247e+09  1.667845e+09  1.660720e+09            31    401630.50
5   1616482800  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.686975e+09  1.686574e+09  1.679542e+09            31    400759.00
6   1616482800  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.678829e+09  1.678429e+09  1.671032e+09            31    399989.50
7   1616482740      MG02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.731507e+09  2.731208e+09  1.310100e+03            15    299725.30
8   1616482740      MG02       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.564829e+09  2.564548e+09  1.222000e+03            15    280925.40
9   1616482800  Tomcat03       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.455197e+09  2.454976e+09  1.089750e+03            16    220551.75
10  1616482740  Tomcat04       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.526058e+09  2.525846e+09  1.089200e+03            15    211311.80
11  1616482740      MG01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.727980e+09  2.727769e+09  6.368000e+02            15    210223.30
12  1616482740  Tomcat02       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.537240e+09  2.537031e+09  1.053800e+03            15    209483.40
13  1616482800  Tomcat03        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.159010e+09  2.158804e+09  1.013750e+03            16    205157.25
14  1616482740  Tomcat04        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.229202e+09  2.229003e+09  1.023500e+03            15    199138.10
15  1616482740      MG01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.560575e+09  2.560379e+09  5.862000e+02            15    196622.30
16  1616482740  Tomcat02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.233769e+09  2.233573e+09  9.792000e+02            15    195847.40
17  1616482740  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.686766e+09  1.686574e+09  1.679542e+09            31    191538.00
18  1616482740  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.668029e+09  1.667845e+09  1.660720e+09            31    183717.50
19  1616482740  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...  9.366594e+08  9.364945e+08  9.295056e+08            31    164865.00, {'number_of_component_kpi_series_examined': 1110, 'number_of_breach_rows_found': 180, 'examined_timestamps_limited': [1616482680, 1616482740, 1616482800]})```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_log if present, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure types
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Timezone object per rule (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Allowed components
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]

# Examined timestamps
examined_timestamps = [1616482680, 1616482740, 1616482800]

# 1-2) Restrict to allowed components for threshold computation, then compute global P95, P5, and sample count per (cmdb_id, log_name)
df_log_allowed = df_log[df_log['cmdb_id'].isin(allowed_components)].copy()

log_thresholds = df_log_allowed.groupby(['cmdb_id','log_name'])['value'].agg(
    global_count='size',
    global_P95=lambda s: float(s.quantile(0.95)),
    global_P5=lambda s: float(s.quantile(0.05))
).reset_index()

number_of_log_series_examined = int(log_thresholds.shape[0])

# 3) Filter rows to the examined timestamps and allowed components
df_log_targets = df_log_allowed[df_log_allowed['timestamp'].isin(examined_timestamps)].copy()

# 4) Merge thresholds into target rows and detect breaches
if df_log_targets.empty:
    result_df_breaches = pd.DataFrame(columns=[
        'timestamp','cmdb_id','log_name','value','global_P95','global_P5','global_count','severity'
    ])
    number_of_breach_rows_found = 0
else:
    df_targets = df_log_targets.merge(log_thresholds, on=['cmdb_id','log_name'], how='left')
    # Mark breaches
    df_targets['above_P95'] = df_targets['value'] > df_targets['global_P95']
    df_targets['below_P5'] = df_targets['value'] < df_targets['global_P5']
    # Compute severity: absolute distance from the violated threshold
    df_targets['severity'] = np.where(
        df_targets['above_P95'],
        (df_targets['value'] - df_targets['global_P95']).abs(),
        np.where(df_targets['below_P5'],
                 (df_targets['global_P5'] - df_targets['value']).abs(),
                 0.0)
    )
    df_breaches = df_targets[(df_targets['above_P95']) | (df_targets['below_P5'])].copy()
    number_of_breach_rows_found = int(df_breaches.shape[0])
    if df_breaches.empty:
        result_df_breaches = pd.DataFrame(columns=[
            'timestamp','cmdb_id','log_name','value','global_P95','global_P5','global_count','severity'
        ])
    else:
        # Select required columns and sort by severity desc, limit top 20
        result_df_breaches = df_breaches[[
            'timestamp','cmdb_id','log_name','value','global_P95','global_P5','global_count','severity'
        ]].sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)

# Summary
summary = {
    'number_of_log_series_examined': number_of_log_series_examined,
    'number_of_breach_rows_found': number_of_breach_rows_found,
    'examined_timestamps': examined_timestamps
}

# Output (DataFrame and summary dict). Use variable names as requested.
result_df_breaches, summary

```
Out[6]:
```
Summary of the log.csv check (plain English):

- Scope: log.csv restricted to allowed components. Global P95/P5 were computed per (cmdb_id, log_name) across the full file before filtering.

- Examined timestamps: [1616482680, 1616482740, 1616482800].

- Counts:
  - number_of_log_series_examined = 18
  - number_of_breach_rows_found = 8

- Breach highlights (top 8 breach rows returned, sorted by severity):
  1. 1616482800 — apache02 — log.row_count = 7.0 (global_P95=1095.0, global_P5=875.5, global_count=31) — far below P5.
  2. 1616482800 — apache01 — log.row_count = 12.0 (global_P95=1085.5, global_P5=868.5, global_count=31) — far below P5.
  3. 1616482800 — Tomcat03 — log.row_count = 10.0 (global_P95=557.5, global_P5=444.0, global_count=31) — far below P5.
  4. 1616482800 — Tomcat02 — log.row_count = 12.0 (global_P95=558.0, global_P5=429.0, global_count=31) — far below P5.
  5. 1616482800 — Tomcat01 — log.row_count = 7.0 (global_P95=563.5, global_P5=423.0, global_count=31) — far below P5.
  6. 1616482800 — Tomcat04 — log.row_count = 5.0 (global_P95=553.0, global_P5=419.5, global_count=31) — far below P5.
  7. 1616482680 — apache02 — log.row_count = 1106.0 (global_P95=1095.0, global_P5=875.5, global_count=31) — slightly above P95.
  8. 1616482680 — Tomcat02 — log.row_count = 560.0 (global_P95=558.0, global_P5=429.0, global_count=31) — slightly above P95.

- Interpretation:
  - The dominant pattern is a sharp drop in log.row_count at timestamp 1616482800 for multiple front-end/app components (apache01/02 and Tomcat01–04), with values far below historical P5 — this suggests either a logging outage/ingestion problem or an actual sharp reduction in generated logs at that time.
  - There are also two earlier small spikes (1616482680) slightly above the historical P95 for apache02 and Tomcat02.
  - Each series used ~31 historical samples to compute thresholds.

- Suggested next checks:
  - Verify log ingestion/forwarding at ~1616482800 (collectors, agents, network).
  - Inspect application status around 1616482680–1616482800 for restarts or log rotation events that could explain the abrupt drop.

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp   cmdb_id       log_name   value  global_P95  global_P5  global_count  severity
0  1616482800  apache02  log.row_count     7.0      1095.0      875.5            31     868.5
1  1616482800  apache01  log.row_count    12.0      1085.5      868.5            31     856.5
2  1616482800  Tomcat03  log.row_count    10.0       557.5      444.0            31     434.0
3  1616482800  Tomcat02  log.row_count    12.0       558.0      429.0            31     417.0
4  1616482800  Tomcat01  log.row_count     7.0       563.5      423.0            31     416.0
5  1616482800  Tomcat04  log.row_count     5.0       553.0      419.5            31     414.5
6  1616482680  apache02  log.row_count  1106.0      1095.0      875.5            31      11.0
7  1616482680  Tomcat02  log.row_count   560.0       558.0      429.0            31       2.0, {'number_of_log_series_examined': 18, 'number_of_breach_rows_found': 8, 'examined_timestamps': [1616482680, 1616482740, 1616482800]})```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing DataFrames if present, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Timezone (UTC+8) object per rule (for reference)
tz = pytz.timezone('Asia/Shanghai')

# Allowed components and target timestamps
allowed_components = ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
                      "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
target_timestamps = [1616482680, 1616482740, 1616482800]

# 1) Ensure global thresholds exist for metric and log (computed across full-file for allowed components)
metric_allowed = df_metric[df_metric['cmdb_id'].isin(allowed_components)].copy()
metric_thresholds = metric_allowed.groupby(['cmdb_id','kpi_name'])['value'].agg(
    global_count='size',
    global_P95=lambda s: float(s.quantile(0.95)),
    global_P5=lambda s: float(s.quantile(0.05))
).reset_index()

log_allowed = df_log[df_log['cmdb_id'].isin(allowed_components)].copy()
log_thresholds = log_allowed.groupby(['cmdb_id','log_name'])['value'].agg(
    global_count='size',
    global_P95=lambda s: float(s.quantile(0.95)),
    global_P5=lambda s: float(s.quantile(0.05))
).reset_index()

# 2) & 3) Filter rows to the three timestamps and allowed components
metric_targets = metric_allowed[metric_allowed['timestamp'].isin(target_timestamps)].copy()
log_targets = log_allowed[log_allowed['timestamp'].isin(target_timestamps)].copy()

# 4) Merge thresholds and detect breaches for metric
if not metric_targets.empty:
    metric_targets = metric_targets.merge(
        metric_thresholds, on=['cmdb_id','kpi_name'], how='left'
    )
    # breach flags and severity
    metric_targets['above_P95'] = metric_targets['value'] > metric_targets['global_P95']
    metric_targets['below_P5'] = metric_targets['value'] < metric_targets['global_P5']
    metric_targets['severity'] = np.where(
        metric_targets['above_P95'],
        (metric_targets['value'] - metric_targets['global_P95']).abs(),
        np.where(metric_targets['below_P5'],
                 (metric_targets['global_P5'] - metric_targets['value']).abs(),
                 0.0)
    )
    # normalize column names for concatenation
    metric_breaches = metric_targets[(metric_targets['above_P95']) | (metric_targets['below_P5'])].copy()
    metric_breaches = metric_breaches.rename(columns={'kpi_name':'name'})
    metric_breaches['source'] = 'metric'
    metric_breaches = metric_breaches[[
        'timestamp','cmdb_id','name','value','global_P95','global_P5','global_count','severity','source'
    ]]
else:
    metric_breaches = pd.DataFrame(columns=[
        'timestamp','cmdb_id','name','value','global_P95','global_P5','global_count','severity','source'
    ])

# 4) Merge thresholds and detect breaches for log
if not log_targets.empty:
    log_targets = log_targets.merge(
        log_thresholds, on=['cmdb_id','log_name'], how='left'
    )
    log_targets['above_P95'] = log_targets['value'] > log_targets['global_P95']
    log_targets['below_P5'] = log_targets['value'] < log_targets['global_P5']
    log_targets['severity'] = np.where(
        log_targets['above_P95'],
        (log_targets['value'] - log_targets['global_P95']).abs(),
        np.where(log_targets['below_P5'],
                 (log_targets['global_P5'] - log_targets['value']).abs(),
                 0.0)
    )
    log_breaches = log_targets[(log_targets['above_P95']) | (log_targets['below_P5'])].copy()
    log_breaches = log_breaches.rename(columns={'log_name':'name'})
    log_breaches['source'] = 'log'
    log_breaches = log_breaches[[
        'timestamp','cmdb_id','name','value','global_P95','global_P5','global_count','severity','source'
    ]]
else:
    log_breaches = pd.DataFrame(columns=[
        'timestamp','cmdb_id','name','value','global_P95','global_P5','global_count','severity','source'
    ])

# Combine breaches from metric and log
breaches = pd.concat([metric_breaches, log_breaches], ignore_index=True)
# If no breaches, prepare empty outputs
if breaches.empty:
    df_comp_summary = pd.DataFrame(columns=[
        'cmdb_id','earliest_breach_timestamp','breached_items_at_earliest_timestamp','severity_score_at_earliest'
    ])
    overall_summary = {
        'overall_earliest_breach_timestamp': None,
        'components_first_breached_at_overall_earliest': [],
        'number_of_components_with_breaches': 0
    }
else:
    # Ensure types
    breaches['timestamp'] = breaches['timestamp'].astype(int)
    breaches['severity'] = breaches['severity'].astype(float)

    # For each component, find earliest breach timestamp among the three
    comp_earliest = breaches.groupby('cmdb_id')['timestamp'].min().reset_index().rename(columns={'timestamp':'earliest_breach_timestamp'})

    # For each component, collect breached items at that earliest timestamp and compute severity score
    records = []
    for _, row in comp_earliest.iterrows():
        cmdb = row['cmdb_id']
        t = int(row['earliest_breach_timestamp'])
        rows_at_t = breaches[(breaches['cmdb_id']==cmdb) & (breaches['timestamp']==t)].copy()
        # prepare list of items (name and value)
        items = []
        for _, r in rows_at_t.iterrows():
            items.append({'name': r['name'], 'value': float(r['value']), 'source': r['source']})
        # severity score = max severity among items at that timestamp
        severity_score = float(rows_at_t['severity'].max())
        records.append({
            'cmdb_id': cmdb,
            'earliest_breach_timestamp': t,
            'breached_items_at_earliest_timestamp': items,
            'severity_score_at_earliest': severity_score,
            'breached_items_count': int(len(items))
        })

    df_comp_summary = pd.DataFrame(records)

    # Sort by earliest timestamp asc, then severity desc, limit top 20
    df_comp_summary = df_comp_summary.sort_values(by=['earliest_breach_timestamp','severity_score_at_earliest'], ascending=[True, False]).head(20).reset_index(drop=True)

    # overall earliest breach timestamp and components that first breached then
    overall_earliest = int(df_comp_summary['earliest_breach_timestamp'].min())
    comps_first = df_comp_summary[df_comp_summary['earliest_breach_timestamp']==overall_earliest]['cmdb_id'].tolist()

    overall_summary = {
        'overall_earliest_breach_timestamp': overall_earliest,
        'components_first_breached_at_overall_earliest': comps_first,
        'number_of_components_with_breaches': int(df_comp_summary.shape[0])
    }

# Display compact results (DataFrame and overall summary)
df_comp_summary, overall_summary

```
Out[7]:
```
Summary of breaches at the three pre-window timestamps (plain English):

- Overall earliest breach timestamp: 1616482680 (UTC+8 context used earlier).
- Number of components that showed at least one breach at these timestamps: 14.
- Components that first breached at 1616482680:
  IG01, Tomcat02, Tomcat03, IG02, apache02, MG02, MG01, Tomcat01, apache01, Redis02, Mysql01, Mysql02, Tomcat04.
- One component first breached at 1616482740: Redis01.

Per-component compact summary (earliest breach among the three timestamps; severity = max absolute deviation from threshold at that timestamp):

- IG01 — earliest_breach_timestamp: 1616482680; breached_items_count: 3; severity ≈ 1.8653e+07.
- Tomcat02 — earliest_breach_timestamp: 1616482680; breached_items_count: 4; severity ≈ 1.2892e+07.
- Tomcat03 — earliest_breach_timestamp: 1616482680; breached_items_count: 15; severity ≈ 432.2.
- IG02 — earliest_breach_timestamp: 1616482680; breached_items_count: 1; severity = 61.0.
- apache02 — earliest_breach_timestamp: 1616482680; breached_items_count: 1 (log.row_count = 1106.0 at that time); severity = 11.0.
- MG02 — earliest_breach_timestamp: 1616482680; breached_items_count: 2; severity = 3.05.
- MG01 — earliest_breach_timestamp: 1616482680; breached_items_count: 6; severity = 3.00.
- Tomcat01 — earliest_breach_timestamp: 1616482680; breached_items_count: 2; severity ≈ 1.5947.
- apache01 — earliest_breach_timestamp: 1616482680; breached_items_count: 1; severity ≈ 0.19095.
- Redis02 — earliest_breach_timestamp: 1616482680; breached_items_count: 2; severity = 0.0405.
- Mysql01 — earliest_breach_timestamp: 1616482680; breached_items_count: 1; severity ≈ 0.03972.
- Mysql02 — earliest_breach_timestamp: 1616482680; breached_items_count: 1; severity ≈ 0.02052.
- Tomcat04 — earliest_breach_timestamp: 1616482680; breached_items_count: 1; severity ≈ 0.00763.
- Redis01 — earliest_breach_timestamp: 1616482740; breached_items_count: 1; severity ≈ 0.63963.

Notes:
- The largest severities (by far) are for IG01 and Tomcat02 (very large deviations from thresholds), followed by Tomcat03 (moderate severity). Many other components show smaller but measurable breaches at the same earliest timestamp 1616482680.
- The output is limited to components that actually breached among the three timestamps and gives the earliest breach time per component and the items and severity at that time.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  earliest_breach_timestamp               breached_items_at_earliest_timestamp  severity_score_at_earliest  breached_items_count
0       IG01                 1616482680  [{'name': 'JVM-Memory_7778_JVM_Memory_HeapMemo...                1.865331e+07                     3
1   Tomcat02                 1616482680  [{'name': 'OSLinux-OSLinux_LOCALDISK_LOCALDISK...                1.289158e+07                     4
2   Tomcat03                 1616482680  [{'name': 'OSLinux-CPU_CPU-0_SingleCpuidle', '...                4.322000e+02                    15
3       IG02                 1616482680  [{'name': 'JVM-Runtime_7778_JVM_JVM_Uptime', '...                6.100000e+01                     1
4   apache02                 1616482680  [{'name': 'log.row_count', 'value': 1106.0, 's...                1.100000e+01                     1
5       MG02                 1616482680  [{'name': 'JVM-Runtime_7779_JVM_JVM_Uptime', '...                3.050000e+00                     2
6       MG01                 1616482680  [{'name': 'JVM-Runtime_7779_JVM_JVM_Uptime', '...                3.000000e+00                     6
7   Tomcat01                 1616482680  [{'name': 'OSLinux-CPU_CPU-2_SingleCpuidle', '...                1.594685e+00                     2
8   apache01                 1616482680  [{'name': 'OSLinux-CPU_CPU-3_SingleCpuUtil', '...                1.909500e-01                     1
9    Redis02                 1616482680  [{'name': 'OSLinux-CPU_CPU_CPULoad', 'value': ...                4.050000e-02                     2
10   Mysql01                 1616482680  [{'name': 'OSLinux-CPU_CPU-1_SingleCpuUtil', '...                3.972000e-02                     1
11   Mysql02                 1616482680  [{'name': 'Mysql-MySQL_3306_Innodb Row Lock Ti...                2.052000e-02                     1
12  Tomcat04                 1616482680  [{'name': 'OSLinux-CPU_CPU_CPUSysTime', 'value...                7.631550e-03                     1
13   Redis01                 1616482740  [{'name': 'OSLinux-CPU_CPU-0_SingleCpuUtil', '...                6.396250e-01                     1, {'overall_earliest_breach_timestamp': 1616482680, 'components_first_breached_at_overall_earliest': ['IG01', 'Tomcat02', 'Tomcat03', 'IG02', 'apache02', 'MG02', 'MG01', 'Tomcat01', 'apache01', 'Redis02', 'Mysql01', 'Mysql02', 'Tomcat04'], 'number_of_components_with_breaches': 14})```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing dataframes/thresholds if present; otherwise load/recompute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
try:
    df_log
except NameError:
    df_log = pd.read_csv('log.csv')

# Ensure timestamps and values typed correctly
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_log['timestamp'] = df_log['timestamp'].astype(int)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Reuse precomputed thresholds if available; otherwise compute them (full-series as required)
try:
    metric_thresholds
except NameError:
    metric_allowed_all = df_metric[df_metric['cmdb_id'].isin(
        ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
         "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
    )].copy()
    metric_thresholds = metric_allowed_all.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_count='size',
        global_P95=lambda s: float(s.quantile(0.95)),
        global_P5=lambda s: float(s.quantile(0.05))
    ).reset_index()

try:
    log_thresholds
except NameError:
    log_allowed_all = df_log[df_log['cmdb_id'].isin(
        ["apache01","apache02","Tomcat01","Tomcat02","Tomcat03","Tomcat04",
         "MG01","MG02","IG01","IG02","Mysql01","Mysql02","Redis01","Redis02"]
    )].copy()
    log_thresholds = log_allowed_all.groupby(['cmdb_id','log_name'])['value'].agg(
        global_count='size',
        global_P95=lambda s: float(s.quantile(0.95)),
        global_P5=lambda s: float(s.quantile(0.05))
    ).reset_index()

# Target components and timestamps
top_components = ['IG01','Tomcat02','Tomcat03']
target_timestamps = [1616482680,1616482740,1616482800]

# Timezone object (UTC+8) per rules (for reference)
tz = pytz.timezone('Asia/Shanghai')

# 1) Metric rows filter and attach thresholds
metric_targets = df_metric[
    (df_metric['cmdb_id'].isin(top_components)) &
    (df_metric['timestamp'].isin(target_timestamps))
].copy()

if not metric_targets.empty:
    metric_targets = metric_targets.merge(
        metric_thresholds, left_on=['cmdb_id','kpi_name'], right_on=['cmdb_id','kpi_name'], how='left'
    )
    # breach flags and severity (absolute deviation from violated boundary)
    metric_targets['above_P95'] = metric_targets['value'] > metric_targets['global_P95']
    metric_targets['below_P5'] = metric_targets['value'] < metric_targets['global_P5']
    metric_targets['severity'] = np.where(
        metric_targets['above_P95'],
        (metric_targets['value'] - metric_targets['global_P95']).abs(),
        np.where(metric_targets['below_P5'],
                 (metric_targets['global_P5'] - metric_targets['value']).abs(),
                 0.0)
    )
    # Keep requested columns only
    metric_rows_df = metric_targets[[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','global_count','above_P95','below_P5','severity'
    ]].copy()
else:
    metric_rows_df = pd.DataFrame(columns=[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','global_count','above_P95','below_P5','severity'
    ])

# Per-component per-timestamp summary for metrics: count of rows and count of breaches
if not metric_rows_df.empty:
    metric_summary = metric_rows_df.groupby(['cmdb_id','timestamp']).agg(
        metric_row_count=('kpi_name','size'),
        metric_breach_count=('severity', lambda s: int((s>0).sum()))
    ).reset_index().sort_values(['cmdb_id','timestamp'])
else:
    metric_summary = pd.DataFrame(columns=['cmdb_id','timestamp','metric_row_count','metric_breach_count'])

# 2) Log rows filter and attach thresholds
log_targets = df_log[
    (df_log['cmdb_id'].isin(top_components)) &
    (df_log['timestamp'].isin(target_timestamps))
].copy()

if not log_targets.empty:
    log_targets = log_targets.merge(
        log_thresholds, left_on=['cmdb_id','log_name'], right_on=['cmdb_id','log_name'], how='left'
    )
    log_targets['above_P95'] = log_targets['value'] > log_targets['global_P95']
    log_targets['below_P5'] = log_targets['value'] < log_targets['global_P5']
    log_targets['severity'] = np.where(
        log_targets['above_P95'],
        (log_targets['value'] - log_targets['global_P95']).abs(),
        np.where(log_targets['below_P5'],
                 (log_targets['global_P5'] - log_targets['value']).abs(),
                 0.0)
    )
    log_rows_df = log_targets[[
        'timestamp','cmdb_id','log_name','value','global_P95','global_P5','global_count','above_P95','below_P5','severity'
    ]].copy()
else:
    log_rows_df = pd.DataFrame(columns=[
        'timestamp','cmdb_id','log_name','value','global_P95','global_P5','global_count','above_P95','below_P5','severity'
    ])

# Per-component per-timestamp summary for logs: count and breach count
if not log_rows_df.empty:
    log_summary = log_rows_df.groupby(['cmdb_id','timestamp']).agg(
        log_row_count=('log_name','size'),
        log_breach_count=('severity', lambda s: int((s>0).sum()))
    ).reset_index().sort_values(['cmdb_id','timestamp'])
else:
    log_summary = pd.DataFrame(columns=['cmdb_id','timestamp','log_row_count','log_breach_count'])

# 3) Limit total returned rows to top 100 (combine metrics+logs if needed). We'll keep separate lists but ensure each is <=100.
metric_rows_out = metric_rows_df.sort_values(by='severity', ascending=False).head(100)
log_rows_out = log_rows_df.sort_values(by='severity', ascending=False).head(100)

# Overall compact summary
overall_summary = {
    'components': top_components,
    'examined_timestamps': target_timestamps,
    'metric_rows_returned': int(metric_rows_out.shape[0]),
    'log_rows_returned': int(log_rows_out.shape[0]),
    'metric_summary_rows': int(metric_summary.shape[0]),
    'log_summary_rows': int(log_summary.shape[0])
}

# Final displayed outputs (compact). Use variable names per instruction.
metric_rows_out, log_rows_out, metric_summary, log_summary, overall_summary

```
Out[8]:
```
Summary of metric + log details for IG01, Tomcat02, Tomcat03 at timestamps [1616482680, 1616482740, 1616482800]:

Scope & returned rows
- Components examined: IG01, Tomcat02, Tomcat03
- Examined timestamps: [1616482680, 1616482740, 1616482800]
- Metric rows returned (top-100 shown): 100
- Log rows returned: 16

Per-component per-timestamp summary (metric_row_count / metric_breach_count)
- IG01
  - 1616482680: 11 metric rows, 3 metric breaches
  - 1616482740: 9 metric rows, 1 metric breach
  - 1616482800: 13 metric rows, 0 metric breaches
- Tomcat02
  - 1616482680: 33 metric rows, 3 metric breaches
  - 1616482740: 104 metric rows, 17 metric breaches
  - 1616482800: 44 metric rows, 4 metric breaches
- Tomcat03
  - 1616482680: 102 metric rows, 15 metric breaches
  - 1616482740: 41 metric rows, 7 metric breaches
  - 1616482800: 80 metric rows, 12 metric breaches

Per-component per-timestamp summary (log_row_count / log_breach_count)
- Tomcat02
  - 1616482680: 3 log rows, 1 breach
  - 1616482740: 3 log rows, 0 breaches
  - 1616482800: 2 log rows, 1 breach
- Tomcat03
  - 1616482680: 3 log rows, 0 breaches
  - 1616482740: 3 log rows, 0 breaches
  - 1616482800: 2 log rows, 1 breach
- IG01
  - no log rows in that small returned set (0 log rows returned for IG01)

Top metric-level breach examples (largest severities)
- IG01 @ 1616482680
  - JVM-Memory_7778_JVM_Memory_HeapMemoryUsed = 8.799224e+07
  - global_P95 = 8.498951e+08, global_P5 = 1.066455e+08
  - Breach: value < global_P5 (large deficit); severity ≈ 1.8653e+07
- Tomcat02 @ 1616482680
  - Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory = 2.699068e+09
  - global_P95 = 2.686177e+09, global_P5 = 2.466088e+09
  - Breach: value > global_P95; severity ≈ 1.2892e+07
- Tomcat03 / Tomcat02 @ 1616482800
  - Several Tomcat Request-processing KPIs exceed global_P95 (values ~1.66–1.68e+09); severities ~0.4e6
- Network counters on Tomcat/MG (NETPacketsIn/Out) also show large absolute deviations at the same timestamps.

Top log-level breach examples
- Tomcat03 @ 1616482800
  - log.row_count = 10.0 (global_P95=557.5, global_P5=444.0) → far below P5; severity = 434.0
- Tomcat02 @ 1616482800
  - log.row_count = 12.0 (global_P95=558.0, global_P5=429.0) → far below P5; severity = 417.0
- Tomcat02 @ 1616482680
  - log.row_count = 560.0 (global_P95=558.0) → slightly above P95; severity = 2.0

Key takeaways
- IG01 shows a pronounced low JVM-heap value at 1616482680 (value well below global P5) — highest single metric severity.
- Tomcat02 shows both a large JVM-free-memory spike at 1616482680 and multiple metric breaches across timestamps (especially many breaches at 1616482740).
- Tomcat03 shows many metric breaches (highest counts at 1616482680 and 1616482800) and a severe drop in log.row_count at 1616482800 (log.row_count far below historical P5).
- In short: large metric anomalies (memory, request-processing, network counters) occur at 1616482680–1616482800 for IG01, Tomcat02, Tomcat03; logs show severe drops for Tomcat02/Tomcat03 at 1616482800 indicating possible logging/ingestion issues or abrupt behavior change.

If you want, I can:
- provide the full list of metric rows for these components/timestamps (not just top-100),
- or expand the log details for IG01 if more log rows exist outside the small returned slice.

The original code execution output of IPython Kernel is also provided below for reference:

(      timestamp   cmdb_id                                           kpi_name         value    global_P95     global_P5  global_count  above_P95  below_P5     severity
1    1616482680      IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.799224e+07  8.498951e+08  1.066455e+08            31      False      True  18653308.00
23   1616482680  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory  2.699068e+09  2.686177e+09  2.466088e+09             4       True     False  12891583.20
420  1616482800  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.668247e+09  1.667845e+09  1.660720e+09            31       True     False    401630.50
340  1616482800  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.678829e+09  1.678429e+09  1.671032e+09            31       True     False    399989.50
409  1616482800  Tomcat03       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.455197e+09  2.454976e+09  1.089750e+03            16       True     False    220551.75
..          ...       ...                                                ...           ...           ...           ...           ...        ...       ...          ...
299  1616482740  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  5.000000e+02  5.000000e+02            31      False     False         0.00
298  1616482740  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  3.000000e+00  8.500000e+00  1.500000e+00            31      False     False         0.00
297  1616482740  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02  1.000000e+02  1.000000e+02            31      False     False         0.00
296  1616482740  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00  0.000000e+00  0.000000e+00            31      False     False         0.00
30   1616482680  Tomcat02  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...  0.000000e+00  0.000000e+00  0.000000e+00            31      False     False         0.00

[100 rows x 10 columns],      timestamp   cmdb_id          log_name   value  global_P95  global_P5  global_count  above_P95  below_P5  severity
15  1616482800  Tomcat03     log.row_count   10.00    557.5000   444.0000            31      False      True     434.0
13  1616482800  Tomcat02     log.row_count   12.00    558.0000   429.0000            31      False      True     417.0
2   1616482680  Tomcat02     log.row_count  560.00    558.0000   429.0000            31       True     False       2.0
0   1616482680  Tomcat02   log.error_count    0.00      0.0000     0.0000            31      False     False       0.0
4   1616482680  Tomcat03  log.gc_pause_max    0.06      0.0900     0.0215            24      False     False       0.0
5   1616482680  Tomcat03     log.row_count  515.00    557.5000   444.0000            31      False     False       0.0
1   1616482680  Tomcat02  log.gc_pause_max    0.03      0.0985     0.0200            24      False     False       0.0
3   1616482680  Tomcat03   log.error_count    0.00      0.0000     0.0000            31      False     False       0.0
7   1616482740  Tomcat02  log.gc_pause_max    0.05      0.0985     0.0200            24      False     False       0.0
6   1616482740  Tomcat02   log.error_count    0.00      0.0000     0.0000            31      False     False       0.0
8   1616482740  Tomcat02     log.row_count  494.00    558.0000   429.0000            31      False     False       0.0
9   1616482740  Tomcat03   log.error_count    0.00      0.0000     0.0000            31      False     False       0.0
11  1616482740  Tomcat03     log.row_count  543.00    557.5000   444.0000            31      False     False       0.0
10  1616482740  Tomcat03  log.gc_pause_max    0.06      0.0900     0.0215            24      False     False       0.0
12  1616482800  Tomcat02   log.error_count    0.00      0.0000     0.0000            31      False     False       0.0
14  1616482800  Tomcat03   log.error_count    0.00      0.0000     0.0000            31      False     False       0.0,     cmdb_id   timestamp  metric_row_count  metric_breach_count
0      IG01  1616482680                11                    3
1      IG01  1616482740                 9                    1
2      IG01  1616482800                13                    0
3  Tomcat02  1616482680                33                    3
4  Tomcat02  1616482740               104                   17
5  Tomcat02  1616482800                44                    4
6  Tomcat03  1616482680               102                   15
7  Tomcat03  1616482740                41                    7
8  Tomcat03  1616482800                80                   12,     cmdb_id   timestamp  log_row_count  log_breach_count
0  Tomcat02  1616482680              3                 1
1  Tomcat02  1616482740              3                 0
2  Tomcat02  1616482800              2                 1
3  Tomcat03  1616482680              3                 0
4  Tomcat03  1616482740              3                 0
5  Tomcat03  1616482800              2                 1, {'components': ['IG01', 'Tomcat02', 'Tomcat03'], 'examined_timestamps': [1616482680, 1616482740, 1616482800], 'metric_rows_returned': 100, 'log_rows_returned': 16, 'metric_summary_rows': 9, 'log_summary_rows': 6})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

# Reuse df_metric and metric_thresholds if present; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Ensure global thresholds for IG01 are computed on the full series (rule: compute thresholds before filtering)
cmdb = 'IG01'
ig_full = df_metric[df_metric['cmdb_id'] == cmdb].copy()
ig_thresholds = ig_full.groupby('kpi_name')['value'].agg(
    global_count='size',
    global_P95=lambda s: float(s.quantile(0.95)) if len(s) > 0 else np.nan,
    global_P5=lambda s: float(s.quantile(0.05)) if len(s) > 0 else np.nan
).reset_index()
# add cmdb_id column for merging
ig_thresholds['cmdb_id'] = cmdb

# Target substrings and timestamps
substr_pattern = r'JVM|Heap|Memory|Uptime'
target_timestamps = [1616482680, 1616482740, 1616482800]

# 1) Filter metric rows for IG01 and kpi_name matching substrings and timestamps
ig_targets = df_metric[
    (df_metric['cmdb_id'] == cmdb) &
    (df_metric['kpi_name'].str.contains(substr_pattern, case=False, regex=True)) &
    (df_metric['timestamp'].isin(target_timestamps))
].copy()

# 2) Merge with global thresholds (already computed)
if not ig_targets.empty:
    ig_targets = ig_targets.merge(
        ig_thresholds[['cmdb_id','kpi_name','global_P95','global_P5','global_count']],
        on=['cmdb_id','kpi_name'],
        how='left'
    )
    ig_targets['above_P95'] = ig_targets['value'] > ig_targets['global_P95']
    ig_targets['below_P5'] = ig_targets['value'] < ig_targets['global_P5']
    # Keep requested columns and limit to 50 rows
    ig_rows_out = ig_targets[[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','above_P95','below_P5'
    ]].sort_values(['timestamp','kpi_name']).head(50).reset_index(drop=True)
else:
    ig_rows_out = pd.DataFrame(columns=[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','above_P95','below_P5'
    ])

# 3) One-line summary about JVM 'Uptime' metrics at 1616482680 vs other examined timestamps
# Find uptime-related kpis among the filtered IG01 set (use case-insensitive match)
uptime_mask_all = ig_full['kpi_name'].str.contains('Uptime', case=False, regex=True)
uptime_kpis = ig_full.loc[uptime_mask_all, 'kpi_name'].unique().tolist()

if len(uptime_kpis) == 0:
    summary_line = "No uptime-related metrics found for IG01."
else:
    decreased_kpis = []
    # Evaluate each uptime kpi using values at the three target timestamps if present
    for kpi in uptime_kpis:
        vals = {}
        for ts in target_timestamps:
            row = ig_targets[(ig_targets['kpi_name']==kpi) & (ig_targets['timestamp']==ts)]
            if not row.empty:
                vals[ts] = float(row['value'].iloc[0])
        # If we have a value at the focal timestamp 1616482680, compare
        focal = 1616482680
        if focal in vals:
            other_vals = [v for t,v in vals.items() if t != focal]
            if other_vals:
                # Decreased if focal < all other values
                if all(vals[focal] < ov for ov in other_vals):
                    decreased_kpis.append(kpi)
            else:
                # If no other examined timestamps present, consider unusually low if below global P5
                gp5 = ig_thresholds.loc[ig_thresholds['kpi_name']==kpi, 'global_P5']
                if not gp5.empty and not np.isnan(gp5.iloc[0]) and vals[focal] < float(gp5.iloc[0]):
                    decreased_kpis.append(kpi)
        else:
            # No focal timestamp value for this kpi in the filtered set — skip
            pass

    if decreased_kpis:
        summary_line = f"Uptime decrease/unusually-low detected at 1616482680 for kpis: {', '.join(decreased_kpis)}."
    else:
        summary_line = "No uptime decrease or unusually-low uptime detected at 1616482680 among examined uptime metrics."

# Display compact outputs: the IG01 JVM-related rows (<=50) and the one-line summary
ig_rows_out, summary_line

```
Out[9]:
```
Summary for IG01 (JVM-related metrics at timestamps 1616482680, 1616482740, 1616482800):

- Rows returned: 21 JVM-related metric rows (filtered by kpi_name containing JVM / Heap / Memory / Uptime) for the three timestamps.

- Notable breaches at 1616482680:
  - JVM-Memory_7778_JVM_Memory_HeapMemoryUsed = 8.7992e+07 which is below its global P5 (≈1.0665e+08). (breach: value < P5)
  - JVM-Runtime_7778_JVM_JVM_Uptime = 1.561724e+06 which is slightly above its global P95 (≈1.561667e+06). (not low; it is high)

- Other JVM-related metrics (HeapMemoryMax, NoHeapMemoryUsed, ThreadCount, JVM CPULoad, etc.) at the three timestamps show no P95/P5 breaches in the returned set.

- One-line summary about uptime:
  - No JVM "Uptime" decrease or unusually-low uptime was detected at 1616482680; uptime at that timestamp is actually slightly above its historical P95.

- Suggested next step (optional):
  - Investigate the low HeapMemoryUsed value at 1616482680 (could indicate an atypically small heap usage reading or post-GC state).

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp cmdb_id                                      kpi_name         value    global_P95     global_P5  above_P95  below_P5
0   1616482680    IG01      JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09  2.040136e+09      False     False
1   1616482680    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  8.799224e+07  8.498951e+08  1.066455e+08      False      True
2   1616482680    IG01   JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865866e+07  5.867203e+07  5.865866e+07      False     False
3   1616482680    IG01               JVM-Runtime_7778_JVM_JVM_Uptime  1.561724e+06  1.561667e+06  1.560102e+06       True     False
4   1616482680    IG01      JVM-Threads_7778_JVM_ThreadCount_Threads  7.000000e+01  7.550000e+01  6.800000e+01      False     False
..         ...     ...                                           ...           ...           ...           ...        ...       ...
16  1616482800    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  6.843968e+08  8.498951e+08  1.066455e+08      False     False
17  1616482800    IG01   JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  5.865866e+07  5.867203e+07  5.865866e+07      False     False
18  1616482800    IG01     JVM-Operating System_7778_JVM_JVM_CPULoad  5.266000e-01  5.756700e-01  4.944100e-01      False     False
19  1616482800    IG01      JVM-Threads_7778_JVM_ThreadCount_Threads  7.200000e+01  7.550000e+01  6.800000e+01      False     False
20  1616482800    IG01  OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc  2.839900e+01  2.843626e+01  2.832136e+01      False     False

[21 rows x 8 columns], 'No uptime decrease or unusually-low uptime detected at 1616482680 among examined uptime metrics.')

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[10]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric and metric_thresholds if in kernel; otherwise load/compute
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# timezone object (UTC+8) per rule (for reference)
tz = pytz.timezone('Asia/Shanghai')

# Ensure global thresholds computed BEFORE filtering (full-series per (cmdb_id,kpi_name))
try:
    metric_thresholds
except NameError:
    metric_thresholds = df_metric.groupby(['cmdb_id','kpi_name'])['value'].agg(
        global_count='size',
        global_P95=lambda s: float(s.quantile(0.95)),
        global_P5=lambda s: float(s.quantile(0.05))
    ).reset_index()

# Target components and timestamps
components = ['Tomcat02','Tomcat03']
target_timestamps = [1616482680,1616482740,1616482800]

# 1) Filter metric rows for the components, timestamps, and kpi_name containing specified substrings (case-sensitive)
substrs = ['JVM','Uptime','JVM_Uptime','JVM-Runtime','JVM_Runtime']
pattern = '(' + '|'.join(substrs) + ')'

tomcat_targets = df_metric[
    (df_metric['cmdb_id'].isin(components)) &
    (df_metric['timestamp'].isin(target_timestamps)) &
    (df_metric['kpi_name'].str.contains(pattern, case=True, regex=True))
].copy()

# 2) Merge with precomputed thresholds and compute breach flags
if not tomcat_targets.empty:
    tomcat_targets = tomcat_targets.merge(metric_thresholds, on=['cmdb_id','kpi_name'], how='left')
    tomcat_targets['above_P95'] = tomcat_targets['value'] > tomcat_targets['global_P95']
    tomcat_targets['below_P5'] = tomcat_targets['value'] < tomcat_targets['global_P5']
    # Keep requested columns
    tomcat_rows_out = tomcat_targets[[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','above_P95','below_P5'
    ]].sort_values(['cmdb_id','timestamp','kpi_name']).head(50).reset_index(drop=True)
else:
    tomcat_rows_out = pd.DataFrame(columns=[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','above_P95','below_P5'
    ])

# 3) Per-component 1-line summary about 'Uptime' decrease at focal timestamp 1616482680
summary_records = []
focal = 1616482680
for comp in components:
    # find uptime-related kpis for this component across the three timestamps in our filtered set
    comp_rows = tomcat_targets[tomcat_targets['cmdb_id']==comp] if not tomcat_targets.empty else pd.DataFrame()
    # identify uptime-like kpis (case-sensitive 'Uptime' or the exact substrings)
    uptime_mask = comp_rows['kpi_name'].str.contains('Uptime|JVM_Uptime|JVM-Runtime|JVM_Runtime', case=True, regex=True) if not comp_rows.empty else pd.Series([], dtype=bool)
    uptime_kpis = comp_rows.loc[uptime_mask, 'kpi_name'].unique().tolist() if not comp_rows.empty else []

    if len(uptime_kpis) == 0:
        summary_text = f"No uptime-related metric found for {comp} among the filtered rows."
    else:
        decreased_kpis = []
        for kpi in uptime_kpis:
            # collect values at the target timestamps for this kpi
            vals = {}
            subset = df_metric[(df_metric['cmdb_id']==comp) & (df_metric['kpi_name']==kpi) & (df_metric['timestamp'].isin(target_timestamps))]
            for ts in target_timestamps:
                row = subset[subset['timestamp']==ts]
                if not row.empty:
                    vals[ts] = float(row['value'].iloc[0])
            if focal in vals:
                other_vals = [v for t,v in vals.items() if t != focal]
                if other_vals:
                    # declare decreased if focal value is strictly less than all other available values
                    if all(vals[focal] < ov for ov in other_vals):
                        decreased_kpis.append(kpi)
                else:
                    # no other timestamp values; consider unusually-low if below global P5
                    gp5_row = metric_thresholds[(metric_thresholds['cmdb_id']==comp) & (metric_thresholds['kpi_name']==kpi)]
                    if not gp5_row.empty:
                        gp5 = float(gp5_row['global_P5'].iloc[0])
                        if vals[focal] < gp5:
                            decreased_kpis.append(kpi)
            # if focal not present, cannot conclude for this kpi
        if decreased_kpis:
            summary_text = f"At {focal}, uptime decreased/unusually-low for: {', '.join(decreased_kpis)}."
        else:
            summary_text = f"No uptime decrease detected at {focal} for {comp} among available uptime metrics."

    summary_records.append({'cmdb_id': comp, 'summary': summary_text})

summary_df = pd.DataFrame(summary_records)

# Final compact outputs: tomcat_rows_out (<=50 rows) and summary_df (one-line per component)
tomcat_rows_out, summary_df

```
Out[10]:
```
Summary of JVM lifecycle checks for Tomcat02 and Tomcat03 at timestamps [1616482680, 1616482740, 1616482800]:

- Tomcat02
  - Uptime: no uptime value present at 1616482680 in the filtered rows; uptime samples at 1616482740 and 1616482800 are above their historical P95 (i.e., normal/high).
  - JVM-related anomalies at 1616482680:
    - Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory = 2.699068e+09 (above global_P95 = 2.686177e+09).
    - Tomcat-MEMORY_7441-MEMORY_JVMMemoryUsedPercent = 35.0 (slightly below global_P5 = 35.3).
  - One-line conclusion: No evidence of an uptime drop / JVM restart at 1616482680; there are memory-related anomalies (high free memory and slightly low used-percent) at that timestamp.

- Tomcat03
  - Uptime values (OSLinux_ZABBIX_Host_Uptime):
    - 1616482680 = 7,681,626
    - 1616482740 = 7,681,686 (above P95)
    - 1616482800 = 7,681,746 (above P95)
  - Interpretation: uptime at 1616482680 is slightly lower than later timestamps (difference ≈ 60 seconds), and the code flagged this as a decrease/unusually-low for Tomcat03 at 1616482680. However, the uptime is not near zero and increases steadily across the three samples — this pattern does not look like a full JVM/process restart (no abrupt drop to small values), more like normal increasing uptime with a minor lower reading at 1616482680.
  - One-line conclusion: A small lower uptime reading occurs at 1616482680 for Tomcat03, but it does not constitute clear evidence of a JVM restart.

Overall conclusion:
- Neither Tomcat02 nor Tomcat03 show definitive JVM restart evidence at 1616482680. Tomcat02 shows notable memory metric anomalies at that timestamp; Tomcat03 shows a small decrease in reported uptime relative to later samples but not a restart-like drop. Recommend checking application logs and JVM GC/ start events if you need to rule out transient lifecycle events.

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp   cmdb_id                                        kpi_name         value    global_P95     global_P5  above_P95  below_P5
0  1616482680  Tomcat02         Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory  2.699068e+09  2.686177e+09  2.466088e+09       True     False
1  1616482680  Tomcat02  Tomcat-MEMORY_7441-MEMORY_JVMMemoryUsedPercent  3.500000e+01  4.055000e+01  3.530000e+01      False      True
2  1616482740  Tomcat02              OSLinux-OSLinux_ZABBIX_Host_Uptime  7.681633e+06  7.681604e+06  7.680031e+06       True     False
3  1616482800  Tomcat02              OSLinux-OSLinux_ZABBIX_Host_Uptime  7.681693e+06  7.681604e+06  7.680031e+06       True     False
4  1616482680  Tomcat03              OSLinux-OSLinux_ZABBIX_Host_Uptime  7.681626e+06  7.681656e+06  7.680028e+06      False     False
5  1616482740  Tomcat03              OSLinux-OSLinux_ZABBIX_Host_Uptime  7.681686e+06  7.681656e+06  7.680028e+06       True     False
6  1616482800  Tomcat03              OSLinux-OSLinux_ZABBIX_Host_Uptime  7.681746e+06  7.681656e+06  7.680028e+06       True     False,     cmdb_id                                            summary
0  Tomcat02  No uptime decrease detected at 1616482680 for ...
1  Tomcat03  At 1616482680, uptime decreased/unusually-low ...)```
```

In [ ]:
# In[11]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Timezone object per rule (UTC+8) for reference
tz = pytz.timezone('Asia/Shanghai')

# Components and substrings to scan
components_to_scan = ["MG01","MG02","Tomcat01","Tomcat02","Tomcat03","Tomcat04","IG01","IG02"]
substrs = ['NETPackets', 'NETInErr', 'NETOutErr', 'NETInErrPrc', 'NETOutErrPrcc']
# build regex pattern for case-sensitive match (pandas default is case=True)
pattern = '(' + '|'.join(substrs) + ')'

# Time window (inclusive)
start_ts = 1616482600
end_ts = 1616482860

# 1) Compute global thresholds (P95, P5, global_count) across the full metric.csv series
# for the specified components BEFORE filtering to the time window
thresholds_net = df_metric[df_metric['cmdb_id'].isin(components_to_scan)].groupby(
    ['cmdb_id','kpi_name']
)['value'].agg(
    global_count='size',
    global_P95=lambda s: float(s.quantile(0.95)),
    global_P5=lambda s: float(s.quantile(0.05))
).reset_index()

# 2) Filter rows to components, kpi_name matching network substrings, and timestamp window
net_rows = df_metric[
    (df_metric['cmdb_id'].isin(components_to_scan)) &
    (df_metric['kpi_name'].str.contains(pattern, case=True, regex=True)) &
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts)
].copy()

# 3) Attach global thresholds to each matching row
if not net_rows.empty:
    net_rows = net_rows.merge(thresholds_net, on=['cmdb_id','kpi_name'], how='left')
    # flags for breaches and severity (absolute deviation from relevant threshold)
    net_rows['above_P95'] = net_rows['value'] > net_rows['global_P95']
    net_rows['below_P5'] = net_rows['value'] < net_rows['global_P5']
    net_rows['severity'] = np.where(
        net_rows['above_P95'],
        (net_rows['value'] - net_rows['global_P95']).abs(),
        np.where(net_rows['below_P5'],
                 (net_rows['global_P5'] - net_rows['value']).abs(),
                 0.0)
    )
    # Select requested columns
    net_rows_sel = net_rows[[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','above_P95','below_P5','severity'
    ]].copy()
else:
    net_rows_sel = pd.DataFrame(columns=[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','above_P95','below_P5','severity'
    ])

# 4) Limit returned metric rows to top 100 by absolute severity (descending)
top_net_rows = net_rows_sel.sort_values(by='severity', ascending=False).head(100).reset_index(drop=True)

# 5) Per-component summary: earliest timestamp in window with any network breach and count of breach rows
if not net_rows_sel.empty:
    # consider only rows that are breaches
    net_breaches = net_rows_sel[(net_rows_sel['above_P95']) | (net_rows_sel['below_P5'])].copy()
    # compute per-component earliest timestamp and count of breach rows
    per_comp = []
    for comp in components_to_scan:
        comp_b = net_breaches[net_breaches['cmdb_id'] == comp]
        if comp_b.empty:
            per_comp.append({
                'cmdb_id': comp,
                'earliest_timestamp_in_window_with_any_network_breach': None,
                'number_of_network_breach_rows_in_window': 0
            })
        else:
            earliest = int(comp_b['timestamp'].min())
            cnt = int(comp_b.shape[0])
            per_comp.append({
                'cmdb_id': comp,
                'earliest_timestamp_in_window_with_any_network_breach': earliest,
                'number_of_network_breach_rows_in_window': cnt
            })
    per_comp_df = pd.DataFrame(per_comp)
else:
    # No matching network rows -> zero breaches for all components
    per_comp_df = pd.DataFrame([{
        'cmdb_id': comp,
        'earliest_timestamp_in_window_with_any_network_breach': None,
        'number_of_network_breach_rows_in_window': 0
    } for comp in components_to_scan])

# Compact meta summary
meta = {
    'components_scanned_count': len(components_to_scan),
    'time_window': {'start_ts': start_ts, 'end_ts': end_ts},
    'matching_network_rows_total': int(net_rows_sel.shape[0]),
    'matching_network_breach_rows_total': int(((net_rows_sel['above_P95']) | (net_rows_sel['below_P5'])).sum()) if not net_rows_sel.empty else 0,
    'returned_metric_rows_limited_to_top': int(top_net_rows.shape[0])
}

# Final outputs (compact): top_net_rows (DataFrame), per_comp_df (DataFrame), meta (dict)
top_net_rows, per_comp_df, meta

```
Out[11]:
```
Summary of network-related KPI scan (components: MG01, MG02, Tomcat01–Tomcat04, IG01, IG02; window 1616482600–1616482860):

- Overall counts
  - Matching network metric rows in window: 56
  - Network breach rows (value > global_P95 or < global_P5): 10

- Components with network breaches (earliest breach timestamp and breach count in window)
  - MG02 — earliest breach at 1616482740; number_of_network_breach_rows_in_window: 2
  - MG01 — earliest breach at 1616482740; number_of_network_breach_rows_in_window: 2
  - Tomcat02 — earliest breach at 1616482740; number_of_network_breach_rows_in_window: 2
  - Tomcat03 — earliest breach at 1616482740; number_of_network_breach_rows_in_window: 2
  - Tomcat04 — earliest breach at 1616482740; number_of_network_breach_rows_in_window: 2

- Components with no network breaches in the window
  - Tomcat01, IG01, IG02 (zero breach rows in the scanned window)

- Top anomaly examples (by absolute deviation / severity)
  - MG02 @ 1616482740 — OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn: value 2.731507e+09 (global_P95 ≈ 2.731208e+09) — large positive deviation.
  - MG02 @ 1616482740 — OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut: value 2.564829e+09 (global_P95 ≈ 2.564548e+09).
  - Tomcat03 @ 1616482800 — OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut: value 2.455197e+09 (global_P95 ≈ 2.454976e+09).
  - Tomcat04 @ 1616482740 — OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut: value 2.526058e+09 (global_P95 ≈ 2.525846e+09).
  - MG01 @ 1616482740 — OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn: value 2.727980e+09 (global_P95 ≈ 2.727769e+09).

- Compact per-component takeaway
  - MG01 / MG02 / Tomcat02 / Tomcat03 / Tomcat04 each show network packet-count anomalies beginning at 1616482740 (two breach rows each).
  - Tomcat01, IG01, IG02 show no network breaches in this window.

If you want, I can return the exact top-100 matching rows (timestamp, cmdb_id, kpi_name, value, global_P95, global_P5, above_P95, below_P5, severity) in a JSON list.

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp   cmdb_id                                      kpi_name         value    global_P95  global_P5  above_P95  below_P5   severity
0   1616482740      MG02   OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.731507e+09  2.731208e+09    1310.10       True     False  299725.30
1   1616482740      MG02  OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.564829e+09  2.564548e+09    1222.00       True     False  280925.40
2   1616482800  Tomcat03  OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.455197e+09  2.454976e+09    1089.75       True     False  220551.75
3   1616482740  Tomcat04  OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.526058e+09  2.525846e+09    1089.20       True     False  211311.80
4   1616482740      MG01   OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.727980e+09  2.727769e+09     636.80       True     False  210223.30
..         ...       ...                                           ...           ...           ...        ...        ...       ...        ...
51  1616482800  Tomcat01       OSLinux-OSLinux_NETWORK_ens160_NETInErr  0.000000e+00  0.000000e+00       0.00      False     False       0.00
52  1616482800  Tomcat01   OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  1.062000e+03  2.282846e+09     943.75      False     False       0.00
53  1616482800  Tomcat01      OSLinux-OSLinux_NETWORK_ens160_NETOutErr  0.000000e+00  0.000000e+00       0.00      False     False       0.00
54  1616482800  Tomcat02  OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc  0.000000e+00  0.000000e+00       0.00      False     False       0.00
55  1616482800  Tomcat01  OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  1.143000e+03  2.603363e+09    1020.25      False     False       0.00

[56 rows x 9 columns],     cmdb_id  earliest_timestamp_in_window_with_any_network_breach  number_of_network_breach_rows_in_window
0      MG01                                       1.616483e+09                                           2
1      MG02                                       1.616483e+09                                           2
2  Tomcat01                                                NaN                                           0
3  Tomcat02                                       1.616483e+09                                           2
4  Tomcat03                                       1.616483e+09                                           2
5  Tomcat04                                       1.616483e+09                                           2
6      IG01                                                NaN                                           0
7      IG02                                                NaN                                           0, {'components_scanned_count': 8, 'time_window': {'start_ts': 1616482600, 'end_ts': 1616482860}, 'matching_network_rows_total': 56, 'matching_network_breach_rows_total': 10, 'returned_metric_rows_limited_to_top': 56})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[12]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing df_metric if present
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')

# Ensure types
df_metric['timestamp'] = df_metric['timestamp'].astype(int)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Timezone object per rule (UTC+8)
tz = pytz.timezone('Asia/Shanghai')

# Components and substrings to search (case-sensitive)
components = ['MG01','MG02']
substrs = ['NETInErr', 'NETOutErr', 'NETInErrPrc', 'NETOutErrPrcc', 'NETInErrPrcc', 'NETOutErrPrc',
           'NETPacketsIn','NETPacketsOut']
pattern = '(' + '|'.join(substrs) + ')'

# Time window (inclusive)
start_ts = 1616482600
end_ts = 1616482860

# 1) Compute global thresholds (P95, P5) and counts across full-series BEFORE filtering
thresholds_mg = df_metric[df_metric['cmdb_id'].isin(components)].groupby(
    ['cmdb_id','kpi_name']
)['value'].agg(
    global_count='size',
    global_P95=lambda s: float(s.quantile(0.95)),
    global_P5=lambda s: float(s.quantile(0.05))
).reset_index()

# 2) Filter rows for components, pattern, and time window
net_mg_rows = df_metric[
    (df_metric['cmdb_id'].isin(components)) &
    (df_metric['kpi_name'].str.contains(pattern, case=True, regex=True)) &
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts)
].copy()

# 3) Merge thresholds and compute flags and severity
if not net_mg_rows.empty:
    net_mg_rows = net_mg_rows.merge(thresholds_mg, on=['cmdb_id','kpi_name'], how='left')
    net_mg_rows['above_P95'] = net_mg_rows['value'] > net_mg_rows['global_P95']
    net_mg_rows['below_P5'] = net_mg_rows['value'] < net_mg_rows['global_P5']
    # severity as absolute difference to the violated boundary (or 0 if no violation)
    net_mg_rows['severity'] = np.where(
        net_mg_rows['above_P95'],
        (net_mg_rows['value'] - net_mg_rows['global_P95']).abs(),
        np.where(net_mg_rows['below_P5'],
                 (net_mg_rows['global_P5'] - net_mg_rows['value']).abs(),
                 0.0)
    )
    # Select requested columns
    net_mg_sel = net_mg_rows[[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','above_P95','below_P5','severity'
    ]].copy()
else:
    net_mg_sel = pd.DataFrame(columns=[
        'timestamp','cmdb_id','kpi_name','value','global_P95','global_P5','above_P95','below_P5','severity'
    ])

# 4) Top 20 matching rows sorted by severity desc
top_mg_rows = net_mg_sel.sort_values(by='severity', ascending=False).head(20).reset_index(drop=True)

# 5) Per-component compact summary: earliest breach timestamp and number of breach rows
per_comp = []
for comp in components:
    comp_rows = net_mg_sel[net_mg_sel['cmdb_id'] == comp]
    breaches = comp_rows[(comp_rows['above_P95']) | (comp_rows['below_P5'])]
    if breaches.empty:
        per_comp.append({
            'cmdb_id': comp,
            'earliest_timestamp_in_window_with_any_network_error_indicator': None,
            'number_of_network_breach_rows_in_window': 0
        })
    else:
        earliest = int(breaches['timestamp'].min())
        cnt = int(breaches.shape[0])
        # also include max severity for context (compact)
        max_sev = float(breaches['severity'].max())
        per_comp.append({
            'cmdb_id': comp,
            'earliest_timestamp_in_window_with_any_network_error_indicator': earliest,
            'number_of_network_breach_rows_in_window': cnt,
            'max_severity_in_window': max_sev
        })

per_comp_df = pd.DataFrame(per_comp)

# Meta summary
meta = {
    'components_checked': components,
    'time_window': {'start_ts': start_ts, 'end_ts': end_ts},
    'matching_rows_total_in_window': int(net_mg_sel.shape[0]),
    'matching_breach_rows_total_in_window': int(((net_mg_sel['above_P95']) | (net_mg_sel['below_P5'])).sum()) if not net_mg_sel.empty else 0,
    'returned_top_rows': int(top_mg_rows.shape[0])
}

# Final compact outputs (DataFrames and meta dict)
top_mg_rows, per_comp_df, meta

```
Out[12]:
```
summary = (
    "Summary (MG01 & MG02 network check, window 1616482600–1616482860):\n"
    "- Total matching network metric rows: 14; total breach rows: 4.\n"
    "- MG02: earliest breach at 1616482740. Breaches: NETPacketsIn (value ~2.731507e+09, severity ~299,725) "
    "and NETPacketsOut (value ~2.564829e+09, severity ~280,925). NETInErr/NETOutErr counters are 0 in this window.\n"
    "- MG01: earliest breach at 1616482740. Breaches: NETPacketsIn (value ~2.727980e+09, severity ~210,223) "
    "and NETPacketsOut (value ~2.560575e+09, severity ~196,622). NETInErr/NETOutErr counters are 0 in this window.\n"
    "- Interpretation: Both MG nodes show large packet-count spikes at 1616482740 without corresponding error-counter increases. "
    "This suggests a high-volume traffic event (or a metric/sampling artifact) rather than obvious interface errors. "
    "Recommended next steps: inspect network traffic/flows and collectors at ~1616482740, check interface stats on MG01/MG02 and upstream switches/routers, and verify metric collection integrity."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp cmdb_id                                      kpi_name         value    global_P95  global_P5  above_P95  below_P5  severity
0   1616482740    MG02   OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.731507e+09  2.731208e+09     1310.1       True     False  299725.3
1   1616482740    MG02  OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.564829e+09  2.564548e+09     1222.0       True     False  280925.4
2   1616482740    MG01   OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn  2.727980e+09  2.727769e+09      636.8       True     False  210223.3
3   1616482740    MG01  OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut  2.560575e+09  2.560379e+09      586.2       True     False  196622.3
4   1616482680    MG01    OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  0.000000e+00  0.000000e+00        0.0      False     False       0.0
5   1616482680    MG01  OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc  0.000000e+00  0.000000e+00        0.0      False     False       0.0
6   1616482740    MG01      OSLinux-OSLinux_NETWORK_ens160_NETOutErr  0.000000e+00  0.000000e+00        0.0      False     False       0.0
7   1616482740    MG01       OSLinux-OSLinux_NETWORK_ens160_NETInErr  0.000000e+00  0.000000e+00        0.0      False     False       0.0
8   1616482680    MG02  OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc  0.000000e+00  0.000000e+00        0.0      False     False       0.0
9   1616482680    MG02    OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  0.000000e+00  0.000000e+00        0.0      False     False       0.0
10  1616482740    MG02      OSLinux-OSLinux_NETWORK_ens160_NETOutErr  0.000000e+00  0.000000e+00        0.0      False     False       0.0
11  1616482740    MG02       OSLinux-OSLinux_NETWORK_ens160_NETInErr  0.000000e+00  0.000000e+00        0.0      False     False       0.0
12  1616482800    MG02    OSLinux-OSLinux_NETWORK_ens160_NETInErrPrc  0.000000e+00  0.000000e+00        0.0      False     False       0.0
13  1616482800    MG02  OSLinux-OSLinux_NETWORK_ens160_NETOutErrPrcc  0.000000e+00  0.000000e+00        0.0      False     False       0.0,   cmdb_id  earliest_timestamp_in_window_with_any_network_error_indicator  number_of_network_breach_rows_in_window  max_severity_in_window
0    MG01                                         1616482740                                                    2                210223.3
1    MG02                                         1616482740                                                    2                299725.3, {'components_checked': ['MG01', 'MG02'], 'time_window': {'start_ts': 1616482600, 'end_ts': 1616482860}, 'matching_rows_total_in_window': 14, 'matching_breach_rows_total_in_window': 4, 'returned_top_rows': 14})```
```